<a href="https://colab.research.google.com/github/jonasknoll57/Bachelorarbeit_Demand-AD/blob/main/V20d_multicity_ready.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# V17b — Transfer-Pipeline Sanity Check: HD → MA

**Zweck:** Debug-/Sanity-Notebook für die Transfer-Pipeline.
Prüft ob das Setup mit den finalen Anomalietypen (ohne Contextuals) methodisch plausibel funktioniert.

**Experimente:**
1. MA Oracle (Target-only 100%)
2. HD → MA Zero-Shot (Strict Pure)
3. HD → MA Zero-Shot (Calibrated / Target-Stats)
4. HD → MA Fine-Tune (5% Target-Train)

**Modelle:** Forecaster (LSTM dir+ZNorm) als Hauptmodell, AE (ZNorm Hour×Station) als Vergleich.

**Anomalietypen:** point_spike, point_drop, collective (keine contextuals).

In [1]:
# ══════════════════════════════════════════════════════════════
# 0a — Colab Setup
# ══════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

Mounted at /content/drive


In [2]:
# ══════════════════════════════════════════════════════════════
# 0b — Imports
# ══════════════════════════════════════════════════════════════
import os, math, json, random, warnings, time, re, glob, copy, pickle
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import poisson
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score, precision_recall_curve,
    classification_report
)
from numpy.lib.stride_tricks import sliding_window_view

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings("ignore")
print(f"TF {tf.__version__}, GPU: {tf.config.list_physical_devices('GPU')}")

TF 2.19.0, GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
# ══════════════════════════════════════════════════════════════
# 0c — V20d Config
# ══════════════════════════════════════════════════════════════
DATA_BASE    = '/content/drive/MyDrive/BA_Colab/data'
CLEANED_BASE = '/content/drive/MyDrive/BA_Colab/cleaned'

RUN_NAME    = 'v20d_spike_only'
RESULTS_DIR = f'{DATA_BASE}/{RUN_NAME}'
MODELS_DIR  = f'{RESULTS_DIR}/models'
PRELOAD_MODELS_DIR = MODELS_DIR

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# Paths
geo_path           = f'{CLEANED_BASE}/geo_information/geo_information.parquet'
station_names_path = f'{DATA_BASE}/station_names/station_names.parquet'

# City-specific demand paths
CITY_DEMAND = {
    "Mannheim":   f'{CLEANED_BASE}/demand/Mannheim/demand_cleaned.parquet',
    "Heidelberg": f'{CLEANED_BASE}/demand/Heidelberg/demand_cleaned.parquet',
}

@dataclass
class V17bConfig:
    aggregation_minutes: int = 60
    train_ratio: float = 0.67
    val_ratio: float = 0.82
    min_events_per_day: float = 3.0
    rolling_days: int = 56
    min_context_obs: int = 20

    # Model architecture
    ae_window_size: int = 24
    ae_latent_dim: int = 32
    ae_layers: int = 2
    ae_dropout: float = 0.10
    ae_epochs: int = 50
    ae_batch_size: int = 2048
    ae_lr: float = 1e-3
    ae_early_stop: int = 8
    eval_batch_size: int = 2048

    # Regime thresholds
    low_demand_q: float = 0.33
    high_demand_q: float = 0.67

    require_contiguous: bool = True
    use_bidirectional: bool = False

    # Features
    ae_features: List[str] = field(default_factory=lambda: [
        "n_lends", "n_returns",
        "hour_sin", "hour_cos", "dow_sin", "dow_cos",
        "month_sin", "month_cos", "is_weekend"
    ])
    ae_score_features: List[str] = field(default_factory=lambda: ["n_lends", "n_returns"])

    # Injection
    injection_rate: float = 0.015
    injection_seed: int = 42

    # Fine-tune
    finetune_ratio: float = 0.05

cfg = V17bConfig()
print(cfg)
print(f"\nResults: {RESULTS_DIR}")
print(f"Models : {MODELS_DIR}")


V17bConfig(aggregation_minutes=60, train_ratio=0.67, val_ratio=0.82, min_events_per_day=3.0, rolling_days=56, min_context_obs=20, ae_window_size=24, ae_latent_dim=32, ae_layers=2, ae_dropout=0.1, ae_epochs=50, ae_batch_size=2048, ae_lr=0.001, ae_early_stop=8, eval_batch_size=2048, low_demand_q=0.33, high_demand_q=0.67, require_contiguous=True, use_bidirectional=False, ae_features=['n_lends', 'n_returns', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'is_weekend'], ae_score_features=['n_lends', 'n_returns'], injection_rate=0.015, injection_seed=42, finetune_ratio=0.05)

Results: /content/drive/MyDrive/BA_Colab/data/v20d_spike_only
Models : /content/drive/MyDrive/BA_Colab/data/v20d_spike_only/models


---
## Shared Functions (aus V17 übernommen)

In [4]:
# ══════════════════════════════════════════════════════════════
# 1 — Aggregation
# ══════════════════════════════════════════════════════════════
def aggregate_station_level(df: pd.DataFrame, minutes: int = 60,
                            add_relative: bool = False) -> pd.DataFrame:
    out = df.copy()
    freq = f"{minutes}min"
    out["time_bin"] = out["timestamp"].dt.floor(freq)

    agg = (
        out.groupby(
            ["station_id", "station_name_id", "station_name", "location_id", "time_bin"],
            as_index=False
        )
        .agg({
            "n_lends": "sum",
            "n_returns": "sum",
            "latitude": "first",
            "longitude": "first"
        })
        .rename(columns={"time_bin": "hour_ts"})
    )

    agg["total_demand"] = agg["n_lends"] + agg["n_returns"]
    agg["net_flow"] = agg["n_returns"] - agg["n_lends"]
    agg["abs_net_flow"] = agg["net_flow"].abs()

    agg["hour"] = agg["hour_ts"].dt.hour
    agg["dow"] = agg["hour_ts"].dt.dayofweek
    agg["month"] = agg["hour_ts"].dt.month
    agg["is_weekend"] = agg["dow"].isin([5, 6]).astype(int)

    agg["hour_sin"] = np.sin(2 * np.pi * agg["hour"] / 24)
    agg["hour_cos"] = np.cos(2 * np.pi * agg["hour"] / 24)
    agg["dow_sin"]  = np.sin(2 * np.pi * agg["dow"] / 7)
    agg["dow_cos"]  = np.cos(2 * np.pi * agg["dow"] / 7)
    agg["month_sin"] = np.sin(2 * np.pi * (agg["month"] - 1) / 12)
    agg["month_cos"] = np.cos(2 * np.pi * (agg["month"] - 1) / 12)

    agg = agg.sort_values(["station_id", "hour_ts"]).reset_index(drop=True)
    return agg

In [5]:
# ══════════════════════════════════════════════════════════════
# 2 — Gap-Fill
# ══════════════════════════════════════════════════════════════
def fill_missing_time_bins(x: pd.DataFrame, minutes: int = 60) -> pd.DataFrame:
    freq = f"{minutes}min"
    parts = []
    x = x.copy().sort_values(["station_id", "hour_ts"])
    x = (
        x.groupby(["station_id", "hour_ts"], as_index=False)
         .agg({
             "station_name_id": "first", "station_name": "first",
             "location_id": "first", "latitude": "first",
             "longitude": "first", "n_lends": "sum", "n_returns": "sum",
         })
    )
    key_cols = ["station_id", "station_name_id", "station_name",
                "location_id", "latitude", "longitude"]

    for sid, g in x.groupby("station_id", sort=False):
        g = g.sort_values("hour_ts").copy()
        full_idx = pd.date_range(g["hour_ts"].min(), g["hour_ts"].max(), freq=freq)
        g = g.set_index("hour_ts").reindex(full_idx)
        g.index.name = "hour_ts"
        g["n_lends"]   = g["n_lends"].fillna(0).astype(int)
        g["n_returns"] = g["n_returns"].fillna(0).astype(int)
        for col in key_cols:
            g[col] = g[col].ffill().bfill()
        parts.append(g.reset_index())

    result = pd.concat(parts, ignore_index=True)
    result["total_demand"] = result["n_lends"] + result["n_returns"]
    result["net_flow"]     = result["n_returns"] - result["n_lends"]
    result["abs_net_flow"] = result["net_flow"].abs()

    result["hour"] = result["hour_ts"].dt.hour
    result["dow"]  = result["hour_ts"].dt.dayofweek
    result["month"] = result["hour_ts"].dt.month
    result["is_weekend"] = result["dow"].isin([5, 6]).astype(int)

    result["hour_sin"]  = np.sin(2 * np.pi * result["hour"] / 24)
    result["hour_cos"]  = np.cos(2 * np.pi * result["hour"] / 24)
    result["dow_sin"]   = np.sin(2 * np.pi * result["dow"] / 7)
    result["dow_cos"]   = np.cos(2 * np.pi * result["dow"] / 7)
    result["month_sin"] = np.sin(2 * np.pi * (result["month"] - 1) / 12)
    result["month_cos"] = np.cos(2 * np.pi * (result["month"] - 1) / 12)

    return result.sort_values(["station_id", "hour_ts"]).reset_index(drop=True)

In [6]:
# ══════════════════════════════════════════════════════════════
# 3 — Station-Filter, Demand-Regime, Train/Val/Test Split
# ══════════════════════════════════════════════════════════════
def prepare_stations_and_splits(df: pd.DataFrame, cfg, city_name=""):
    n_days = (df["hour_ts"].max() - df["hour_ts"].min()).days + 1
    min_events = int(n_days * cfg.min_events_per_day)

    station_volume = df.groupby("station_id")["total_demand"].sum()
    active_ids = station_volume[station_volume >= min_events].index.tolist()
    df = df[df["station_id"].isin(active_ids)].copy()

    station_profile = (
        df.groupby(["station_id", "station_name"], as_index=False)
          .agg(
              avg_total_demand_h=("total_demand", "mean"),
              avg_lends_h=("n_lends", "mean"),
              avg_returns_h=("n_returns", "mean"),
              latitude=("latitude", "first"),
              longitude=("longitude", "first")
          )
    )
    q1 = station_profile["avg_total_demand_h"].quantile(cfg.low_demand_q)
    q2 = station_profile["avg_total_demand_h"].quantile(cfg.high_demand_q)
    station_profile["demand_regime"] = station_profile["avg_total_demand_h"].apply(
        lambda x: "low" if x <= q1 else ("mid" if x <= q2 else "high")
    )
    df = df.merge(
        station_profile[["station_id", "demand_regime", "avg_total_demand_h"]],
        on="station_id", how="left"
    )

    t_min = df["hour_ts"].min()
    t_max = df["hour_ts"].max()
    total_h = (t_max - t_min).total_seconds() / 3600

    train_end = t_min + pd.Timedelta(hours=int(total_h * cfg.train_ratio))
    val_end   = t_min + pd.Timedelta(hours=int(total_h * cfg.val_ratio))

    cn = f" ({city_name})" if city_name else ""
    print(f"  Aktive Stationen{cn}: {df['station_id'].nunique()}")
    print(f"  Regime: {station_profile['demand_regime'].value_counts().to_dict()}")
    print(f"  Zeitraum: {t_min.date()} bis {t_max.date()} ({n_days} Tage)")
    print(f"  Split: Train < {train_end.date()}, Val < {val_end.date()}, Test ab {val_end.date()}")

    return df, station_profile, train_end, val_end

In [7]:
# ══════════════════════════════════════════════════════════════
# 4 — Rolling Context z-Scores + Poisson + ECDF + Labels
# ══════════════════════════════════════════════════════════════
TARGETS = ["n_lends", "n_returns", "net_flow", "total_demand"]
COUNT_TARGETS = ["n_lends", "n_returns", "total_demand"]

def add_context_keys(x: pd.DataFrame) -> pd.DataFrame:
    x = x.copy()
    x["ctx_hour"] = x["hour_ts"].dt.hour
    x["ctx_dow"]  = x["hour_ts"].dt.dayofweek
    return x

def rolling_context_scores_vectorized(
    full_df: pd.DataFrame, target: str,
    rolling_days: int = 56, min_obs: int = 20
) -> pd.DataFrame:
    x = full_df.copy().sort_values(["station_id", "hour_ts"]).reset_index(drop=True)
    mu_col    = f"{target}_mu_ctx_roll"
    sd_col    = f"{target}_sd_ctx_roll"
    score_col = f"{target}_z_ctx_roll"

    n_slots = max(rolling_days // 7, 4)
    main_window = n_slots
    main_minp   = min(min_obs, main_window)

    grouped = x.groupby(["station_id", "ctx_hour", "ctx_dow"])[target]
    rolling_mean = grouped.transform(
        lambda s: s.shift(1).rolling(window=main_window, min_periods=main_minp).mean()
    )
    rolling_std = grouped.transform(
        lambda s: s.shift(1).rolling(window=main_window, min_periods=main_minp).std(ddof=0)
    )

    fb1_window = n_slots * 2
    fb1_minp   = min(min_obs, fb1_window)
    grouped_sh = x.groupby(["station_id", "ctx_hour"])[target]
    fb1_mean = grouped_sh.transform(
        lambda s: s.shift(1).rolling(window=fb1_window, min_periods=fb1_minp).mean()
    )
    fb1_std = grouped_sh.transform(
        lambda s: s.shift(1).rolling(window=fb1_window, min_periods=fb1_minp).std(ddof=0)
    )

    fb2_window = n_slots * 4
    fb2_minp   = min(min_obs, fb2_window)
    grouped_s = x.groupby(["station_id"])[target]
    fb2_mean = grouped_s.transform(
        lambda s: s.shift(1).rolling(window=fb2_window, min_periods=fb2_minp).mean()
    )
    fb2_std = grouped_s.transform(
        lambda s: s.shift(1).rolling(window=fb2_window, min_periods=fb2_minp).std(ddof=0)
    )

    mu = rolling_mean.copy()
    sd = rolling_std.copy()
    mask1 = mu.isna()
    mu = mu.where(~mask1, fb1_mean)
    sd = sd.where(~mask1, fb1_std)
    mask2 = mu.isna()
    mu = mu.where(~mask2, fb2_mean)
    sd = sd.where(~mask2, fb2_std)
    sd = sd.clip(lower=1e-6)
    z = (x[target] - mu) / sd

    x[mu_col]    = mu
    x[sd_col]    = sd
    x[score_col] = z
    return x


def add_rolling_poisson_scores_vectorized(
    full_df: pd.DataFrame, target: str,
    rolling_days: int = 56, min_obs: int = 20
) -> pd.DataFrame:
    x = full_df.copy().sort_values(["station_id", "hour_ts"]).reset_index(drop=True)
    lam_col       = f"{target}_lambda_ctx_roll"
    score_col     = f"{target}_poisson_upper_score"
    score_low_col = f"{target}_poisson_lower_score"
    mu_col        = f"{target}_mu_ctx_roll"

    if mu_col not in x.columns:
        raise ValueError(f"{mu_col} muss zuerst berechnet werden")

    x[lam_col] = x[mu_col].clip(lower=1e-6)
    vals = x[target].values
    lams = x[lam_col].values

    with np.errstate(divide='ignore', invalid='ignore'):
        tail_p = poisson.sf(vals.astype(float) - 1, lams.astype(float))
        score  = -np.log10(np.clip(tail_p, 1e-12, None))
        lower_p   = poisson.cdf(vals.astype(float), lams.astype(float))
        score_low = -np.log10(np.clip(lower_p, 1e-12, None))

    mask_nan = np.isnan(lams)
    score[mask_nan]     = np.nan
    score_low[mask_nan] = np.nan

    x[score_col]     = score
    x[score_low_col] = score_low
    return x


def percentile_score(train_vals: np.ndarray, values: np.ndarray) -> np.ndarray:
    train_vals = np.asarray(train_vals, dtype=float)
    values     = np.asarray(values, dtype=float)
    train_vals = train_vals[np.isfinite(train_vals)]
    if len(train_vals) == 0:
        return np.full(len(values), np.nan, dtype=float)
    train_sorted = np.sort(train_vals)
    ranks = np.searchsorted(train_sorted, values, side="right")
    return ranks / len(train_sorted)

def calibrate_scores_by_station(
    full_df: pd.DataFrame, train_mask: pd.Series, raw_score_cols: List[str]
) -> pd.DataFrame:
    x = full_df.copy()
    for col in raw_score_cols:
        if col not in x.columns:
            continue
        pct_col = f"{col}_pct_station"
        x[pct_col] = np.nan
        for sid, grp in x.groupby("station_id", sort=False):
            idx_all   = grp.index
            idx_train = grp.index[train_mask.loc[idx_all]]
            train_vals = x.loc[idx_train, col].to_numpy(dtype=float) if len(idx_train) > 0 else np.array([])
            vals       = x.loc[idx_all, col].to_numpy(dtype=float)
            x.loc[idx_all, pct_col] = percentile_score(train_vals, vals)
    return x

def build_eval_labels_calibrated(x: pd.DataFrame) -> pd.DataFrame:
    y = x.copy()
    y["label_eval"] = "normal"

    z_up = "total_demand_score_upper_pct_station"
    z_low = "total_demand_score_lower_pct_station"
    p_up  = "total_demand_poisson_upper_score_pct_station"
    p_low = "total_demand_poisson_lower_score_pct_station"

    required = [z_up, z_low, p_up, p_low]
    missing = [c for c in required if c not in y.columns]
    if missing:
        raise ValueError(f"Fehlende Spalten: {missing}")

    td = y["total_demand"].fillna(0)
    mu = y["total_demand_mu_ctx_roll"]

    abs_min_high    = np.maximum(5, np.ceil(mu + 2)).fillna(5)
    abs_min_low_ref = np.maximum(3, np.ceil(mu)).fillna(3)

    high_strong    = ((y[z_up] >= 0.995) | (y[p_up] >= 0.995)) & (td >= abs_min_high)
    high_consensus = ((y[z_up] >= 0.99) & (y[p_up] >= 0.99)) & (td >= abs_min_high)
    cond_high = high_strong | high_consensus

    low_possible  = mu >= abs_min_low_ref
    low_strong    = ((y[z_low] >= 0.999) | (y[p_low] >= 0.999)) & low_possible
    low_consensus = ((y[z_low] >= 0.995) & (y[p_low] >= 0.995)) & low_possible
    cond_low = (low_strong | low_consensus) & ~cond_high

    grau_high = ((y[z_up] >= 0.99) | (y[p_up] >= 0.99)) & ~cond_high & ~cond_low
    grau_low  = ((y[z_low] >= 0.99) | (y[p_low] >= 0.99)) & ~cond_high & ~cond_low & ~grau_high

    y.loc[grau_high, "label_eval"] = "grauzone_high"
    y.loc[grau_low,  "label_eval"] = "grauzone_low"
    y.loc[cond_high, "label_eval"] = "anomal_high"
    y.loc[cond_low,  "label_eval"] = "anomal_low"
    return y


def run_statistical_pipeline(df: pd.DataFrame, cfg, train_end):
    print("  [1/5] Kontext-Keys...")
    df = add_context_keys(df)
    print("  [2/5] Rolling z-Scores...")
    for tgt in TARGETS:
        df = rolling_context_scores_vectorized(
            df, target=tgt,
            rolling_days=cfg.rolling_days, min_obs=cfg.min_context_obs
        )
    for tgt in TARGETS:
        zc = f"{tgt}_z_ctx_roll"
        df[f"{tgt}_score_upper"] = df[zc]
        df[f"{tgt}_score_lower"] = -df[zc]

    print("  [3/5] Poisson-Tail-Scores...")
    for tgt in COUNT_TARGETS:
        df = add_rolling_poisson_scores_vectorized(
            df, target=tgt,
            rolling_days=cfg.rolling_days, min_obs=cfg.min_context_obs
        )

    print("  [4/5] ECDF-Kalibrierung...")
    raw_score_cols = []
    for tgt in TARGETS:
        raw_score_cols += [f"{tgt}_score_upper", f"{tgt}_score_lower"]
    for tgt in COUNT_TARGETS:
        raw_score_cols += [f"{tgt}_poisson_upper_score", f"{tgt}_poisson_lower_score"]

    train_mask = df["hour_ts"] < train_end
    df = calibrate_scores_by_station(df, train_mask, raw_score_cols)

    print("  [5/5] Labels...")
    df = build_eval_labels_calibrated(df)

    anomaly_rate = df["label_eval"].isin(["anomal_high", "anomal_low"]).mean()
    print(f"  Label-Verteilung:")
    print(f"    {df['label_eval'].value_counts().to_dict()}")
    print(f"    Anomalie-Rate: {anomaly_rate:.4f} ({anomaly_rate*100:.2f}%)")
    return df

In [8]:
# ══════════════════════════════════════════════════════════════
# 5 — Sequenzbuilder (Window-Level Labels)
# ══════════════════════════════════════════════════════════════
def make_sequences_with_window_labels(
    x: pd.DataFrame, feature_cols: List[str], window_size: int,
    synth_label_col: str = "synth_label",
    synth_type_col: str = "synth_type",
    require_contiguous: bool = True,
    agg_minutes: int = 60
) -> Tuple[np.ndarray, pd.DataFrame]:
    X_parts, meta_parts = [], []
    expected_ns = pd.Timedelta(minutes=agg_minutes).value

    for sid, g in x.groupby("station_id", sort=False):
        g = g.sort_values("hour_ts").reset_index(drop=True)
        n = len(g)
        if n < window_size:
            continue

        vals = g[feature_cols].to_numpy(dtype=np.float32)
        win  = sliding_window_view(vals, window_shape=window_size, axis=0)
        win  = np.moveaxis(win, -1, 1)

        valid_mask = np.ones(n - window_size + 1, dtype=bool)

        if require_contiguous:
            ts_int = pd.to_datetime(g["hour_ts"]).astype("int64").to_numpy()
            diffs  = np.diff(ts_int)
            step_ok = (diffs == expected_ns).astype(np.int8)
            if window_size > 1:
                ok_sums = np.convolve(step_ok, np.ones(window_size-1, dtype=np.int32), mode="valid")
                valid_mask = (ok_sums == (window_size - 1))

        if not valid_mask.any():
            continue

        end_indices = np.arange(window_size - 1, n)[valid_mask]
        X_parts.append(win[valid_mask])

        meta_chunk = g.iloc[end_indices].copy()

        synth_arr = g[synth_label_col].to_numpy()
        type_arr  = g[synth_type_col].to_numpy()

        window_labels, window_types, window_counts = [], [], []
        for end_idx in end_indices:
            start_idx = end_idx - window_size + 1
            window_synth = synth_arr[start_idx:end_idx + 1]
            window_type  = type_arr[start_idx:end_idx + 1]

            has_synth = int(window_synth.max())
            n_synth   = int(window_synth.sum())

            if has_synth:
                synth_positions = np.where(window_synth == 1)[0]
                wtype = window_type[synth_positions[-1]]
            else:
                wtype = "none"

            window_labels.append(has_synth)
            window_types.append(wtype)
            window_counts.append(n_synth)

        meta_chunk["window_synth_label"] = window_labels
        meta_chunk["window_synth_type"]  = window_types
        meta_chunk["n_synth_in_window"]  = window_counts
        meta_parts.append(meta_chunk)

    if X_parts:
        X    = np.concatenate(X_parts, axis=0)
        meta = pd.concat(meta_parts, axis=0).reset_index(drop=True)
    else:
        X    = np.empty((0, window_size, len(feature_cols)), dtype=np.float32)
        meta = pd.DataFrame()

    return X, meta

In [9]:
# ══════════════════════════════════════════════════════════════
# 6 — Model Builders (AE + Forecaster)
# ══════════════════════════════════════════════════════════════
def build_lstm_autoencoder(
    n_input_features: int,
    window_size: int,
    latent_dim: int = 32,
    n_layers: int = 2,
    dropout: float = 0.1,
    bidirectional: bool = False,
    n_output_features: Optional[int] = None,
) -> keras.Model:
    n_output_features = n_output_features or n_input_features
    inputs = keras.Input(shape=(window_size, n_input_features), name="encoder_input")
    x = inputs
    for i in range(n_layers):
        return_sequences = (i < n_layers - 1)
        lstm = layers.LSTM(
            latent_dim, return_sequences=return_sequences,
            dropout=dropout, name=f"encoder_lstm_{i+1}"
        )
        x = layers.Bidirectional(lstm, name=f"encoder_bi_{i+1}")(x) if bidirectional else lstm(x)
    latent = x
    x = layers.RepeatVector(window_size, name="repeat_vector")(latent)
    for i in range(n_layers):
        lstm = layers.LSTM(
            latent_dim, return_sequences=True,
            dropout=dropout, name=f"decoder_lstm_{i+1}"
        )
        x = layers.Bidirectional(lstm, name=f"decoder_bi_{i+1}")(x) if bidirectional else lstm(x)
    outputs = layers.TimeDistributed(
        layers.Dense(n_output_features), name="output_dense"
    )(x)
    return keras.Model(inputs, outputs, name="lstm_autoencoder")


def build_lstm_forecaster(
    n_input_features: int,
    n_output_features: int,
    context_steps: int = 23,
    latent_dim: int = 32,
    n_layers: int = 2,
    dropout: float = 0.1,
) -> keras.Model:
    inputs = keras.Input(shape=(context_steps, n_input_features), name="forecast_input")
    x = inputs
    for i in range(n_layers):
        return_sequences = (i < n_layers - 1)
        x = layers.LSTM(
            latent_dim, return_sequences=return_sequences,
            dropout=dropout, name=f"forecast_lstm_{i+1}"
        )(x)
    outputs = layers.Dense(n_output_features, name="forecast_output")(x)
    return keras.Model(inputs, outputs, name="lstm_forecaster")


def train_model_generic(model, X_train, Y_train, X_val, Y_val,
                        epochs, lr, batch_size, early_stop, verbose=1):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr, clipnorm=1.0),
        loss="mse"
    )
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=early_stop,
            restore_best_weights=True, verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=4, verbose=1
        )
    ]
    history = model.fit(
        X_train, Y_train,
        validation_data=(X_val, Y_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks,
        verbose=verbose
    )
    return model, history

In [10]:
# ══════════════════════════════════════════════════════════════
# 7 — Scoring Helpers (AE + Forecaster)
# ══════════════════════════════════════════════════════════════
def predict_in_batches(model, X, batch_size=2048):
    preds = []
    for i in range(0, len(X), batch_size):
        preds.append(model.predict(X[i:i + batch_size], verbose=0))
    return np.concatenate(preds, axis=0) if preds else np.empty((0, 0, 0), dtype=np.float32)


def compute_reconstruction_scores(X, X_hat, input_features, score_features,
                                  mode="last_step_mean"):
    score_idx = [input_features.index(c) for c in score_features]
    err = (X - X_hat) ** 2
    err_score = err[:, :, score_idx]
    if mode == "last_step_mean":
        return err_score[:, -1, :].mean(axis=1)
    elif mode == "window_mean":
        return err_score.mean(axis=(1, 2))
    elif mode == "max_over_features":
        return err_score[:, -1, :].max(axis=1)
    else:
        raise ValueError(f"Unbekannter mode: {mode}")


def compute_forecast_scores(model, X_input, X_actual_last_step, score_features,
                            ae_features, mode="directional"):
    X_pred = predict_in_batches(model, X_input, batch_size=2048)
    score_idx = [ae_features.index(f) for f in score_features]
    X_actual = X_actual_last_step[:, score_idx]
    err = (X_pred - X_actual) ** 2
    if mode == "mean":
        return err.mean(axis=1)
    elif mode == "max":
        return err.max(axis=1)
    elif mode == "directional":
        abs_err = np.abs(X_pred - X_actual)
        denom = np.maximum(np.maximum(np.abs(X_actual), np.abs(X_pred)), 1e-6)
        rel_err = abs_err / denom
        return 0.5 * abs_err.mean(axis=1) + 0.5 * rel_err.mean(axis=1)
    else:
        raise ValueError(f"Unbekannter mode: {mode}")

In [11]:
# ══════════════════════════════════════════════════════════════
# 8 — Z-Normalisierung (Hour + Hour×Station)
# ══════════════════════════════════════════════════════════════
def znorm_score_by_hour(meta_df, raw_score_col, val_start, test_start,
                        new_col="score_znorm_hour"):
    meta = meta_df.copy()
    val_normal = (
        (meta["hour_ts"] >= val_start) &
        (meta["hour_ts"] < test_start) &
        (meta["synth_label"] == 0)
    )
    hour_col = meta["hour_ts"].dt.hour
    mu_per_hour = {}
    std_per_hour = {}
    for h in range(24):
        vals = meta.loc[val_normal & (hour_col == h), raw_score_col].dropna()
        mu_per_hour[h]  = vals.mean() if len(vals) > 0 else 0.0
        std_per_hour[h] = vals.std()  if len(vals) > 1 else 1.0

    hours = hour_col.values
    raw   = meta[raw_score_col].values
    znorm = np.zeros_like(raw)
    for i in range(len(raw)):
        h = hours[i]
        znorm[i] = (raw[i] - mu_per_hour[h]) / max(std_per_hour[h], 1e-8)
    meta[new_col] = znorm
    return meta, mu_per_hour, std_per_hour


def znorm_score_by_hour_station(meta_df, raw_score_col, val_start, test_start,
                                new_col="score_znorm_hour_station"):
    meta = meta_df.copy()
    val_normal = (
        (meta["hour_ts"] >= val_start) &
        (meta["hour_ts"] < test_start) &
        (meta["synth_label"] == 0)
    )
    hour_col = meta["hour_ts"].dt.hour
    meta["_hour_tmp"] = hour_col

    stats = (
        meta[val_normal]
        .groupby(["station_id", "_hour_tmp"])[raw_score_col]
        .agg(["mean", "std", "count"])
        .reset_index()
    )
    global_stats = (
        meta[val_normal]
        .groupby("_hour_tmp")[raw_score_col]
        .agg(["mean", "std"])
        .reset_index()
        .rename(columns={"mean": "g_mean", "std": "g_std"})
    )
    stats = stats.merge(global_stats, on="_hour_tmp", how="left")
    stats["use_mean"] = np.where(stats["count"] >= 10, stats["mean"], stats["g_mean"])
    stats["use_std"]  = np.where(stats["count"] >= 10, stats["std"],  stats["g_std"])
    stats["use_std"]  = stats["use_std"].clip(lower=1e-8)

    meta = meta.merge(
        stats[["station_id", "_hour_tmp", "use_mean", "use_std"]],
        on=["station_id", "_hour_tmp"], how="left"
    )
    meta["use_mean"] = meta["use_mean"].fillna(0.0)
    meta["use_std"]  = meta["use_std"].fillna(1.0).clip(lower=1e-8)
    meta[new_col] = (meta[raw_score_col] - meta["use_mean"]) / meta["use_std"]
    meta = meta.drop(columns=["_hour_tmp", "use_mean", "use_std"])
    return meta

In [12]:
# ══════════════════════════════════════════════════════════════
# 9 — Synthetic Injection (V17b: OHNE Contextuals)
# ══════════════════════════════════════════════════════════════
# Original V17 balanced probs: [spike=0.35, drop=0.30, ctx=0.25, coll=0.10]
# Ohne contextuals, renormiert auf 3 Typen:
# spike = 0.35/(0.35+0.30+0.10) = 0.467
# drop  = 0.30/(0.35+0.30+0.10) = 0.400
# coll  = 0.10/(0.35+0.30+0.10) = 0.133

V17B_INJECTION_PROBS = {
    "point_spike": 1.0
}
V17B_COLLECTIVE_BLOCK_LEN = (2, 4)

def inject_synthetic_anomalies_v17b(
    df: pd.DataFrame,
    inject_start: pd.Timestamp,
    injection_rate: float = 0.015,
    seed: int = 42,
    verbose: bool = True
) -> pd.DataFrame:
    """
    Injection ohne contextuals.
    Gibt zusätzlich requested vs actually injected counts zurück.
    """
    probs = list(V17B_INJECTION_PROBS.values())
    type_names = list(V17B_INJECTION_PROBS.keys())
    block_min, block_max = V17B_COLLECTIVE_BLOCK_LEN

    rng = np.random.RandomState(seed)
    out = df.copy()

    out["original_n_lends"]   = out["n_lends"].copy()
    out["original_n_returns"] = out["n_returns"].copy()
    out["synth_label"] = 0
    out["synth_type"]  = "none"

    inject_mask = out["hour_ts"] >= inject_start
    inject_idx  = out[inject_mask].index.to_numpy()

    if len(inject_idx) == 0:
        print("  WARNUNG: Keine Daten ab inject_start!")
        return out

    n_inject = max(1, int(len(inject_idx) * injection_rate))
    inject_df = out.loc[inject_idx]
    has_demand = inject_df[inject_df["total_demand"] > 0].index.to_numpy()

    if len(has_demand) < n_inject:
        n_inject = len(has_demand)

    chosen_idx = rng.choice(has_demand, size=n_inject, replace=False)
    types = rng.choice(type_names, size=n_inject, p=probs)

    injected_indices = set()

    # Track requested vs actually injected
    requested_counts = {t: 0 for t in type_names}
    injected_counts  = {t: 0 for t in type_names}

    for idx, anom_type in zip(chosen_idx, types):
        requested_counts[anom_type] += 1
        if idx in injected_indices:
            continue

        row = out.loc[idx]

        if anom_type == "point_spike":
            factor = rng.uniform(2.0, 4.0)
            out.loc[idx, "n_lends"]   = int(row["n_lends"] * factor) + rng.randint(1, 4)
            out.loc[idx, "n_returns"] = int(row["n_returns"] * factor) + rng.randint(1, 4)
            out.loc[idx, "synth_label"] = 1
            out.loc[idx, "synth_type"]  = "point_spike"
            injected_indices.add(idx)
            injected_counts["point_spike"] += 1

        elif anom_type == "point_drop":
            regime = row.get("demand_regime", "low")
            if regime == "high" and row["total_demand"] >= 8:
                do_drop = True
            elif regime == "mid" and row["total_demand"] >= 5:
                do_drop = True
            else:
                do_drop = False

            if do_drop:
                out.loc[idx, "n_lends"]   = rng.randint(0, 2)
                out.loc[idx, "n_returns"] = rng.randint(0, 2)
                out.loc[idx, "synth_label"] = 1
                out.loc[idx, "synth_type"]  = "point_drop"
                injected_indices.add(idx)
                injected_counts["point_drop"] += 1

        elif anom_type == "collective":
            block_len = rng.randint(block_min, block_max + 1)
            sid = row["station_id"]
            station_inject = out[
                (out["station_id"] == sid) & inject_mask
            ].sort_values("hour_ts")

            if idx in station_inject.index:
                pos = station_inject.index.get_loc(idx)
                if pos + block_len <= len(station_inject):
                    block_idx = station_inject.index[pos:pos + block_len]
                    factor = rng.uniform(2.5, 5.0)
                    block_actually = 0
                    for bidx in block_idx:
                        if bidx not in injected_indices:
                            out.loc[bidx, "n_lends"]   = int(out.loc[bidx, "original_n_lends"] * factor) + rng.randint(1, 5)
                            out.loc[bidx, "n_returns"] = int(out.loc[bidx, "original_n_returns"] * factor) + rng.randint(1, 5)
                            out.loc[bidx, "synth_label"] = 1
                            out.loc[bidx, "synth_type"]  = "collective"
                            injected_indices.add(bidx)
                            block_actually += 1
                    injected_counts["collective"] += block_actually

    # Update derived columns
    out["total_demand"]  = out["n_lends"] + out["n_returns"]
    out["net_flow"]      = out["n_returns"] - out["n_lends"]
    out["abs_net_flow"]  = out["net_flow"].abs()

    n_injected = out["synth_label"].sum()
    inject_total = inject_mask.sum()

    if verbose:
        print(f"\n  Synthetic Injection (V17b — ohne Contextuals):")
        print(f"    Inject-Zeilen gesamt:  {inject_total:,}")
        print(f"    Injizierte Punkte:     {n_injected:,} ({n_injected/inject_total*100:.2f}%)")
        print(f"\n    Requested vs Actually Injected:")
        for t in type_names:
            print(f"      {t:15s}  requested={requested_counts[t]:5d}  injected={injected_counts[t]:5d}")
        print(f"\n    Effective rates (of inject region):")
        for t in type_names:
            rate = injected_counts[t] / inject_total * 100
            print(f"      {t:15s}  {rate:.3f}%")

    return out

In [13]:
# ══════════════════════════════════════════════════════════════
# 10 — City Data Pipeline
# ══════════════════════════════════════════════════════════════
def prepare_city_data(demand_path: str, geo_path: str, station_names_path: str,
                      cfg, city_name: str):
    print(f"\n{'='*70}")
    print(f"  DATEN-PIPELINE: {city_name}")
    print(f"{'='*70}")

    demand = pd.read_parquet(demand_path)
    geo    = pd.read_parquet(geo_path)
    snames = pd.read_parquet(station_names_path)
    snames = snames.rename(columns={'id': 'station_name_id', 'name': 'station_name'})

    demand = demand.merge(snames[['station_name_id', 'station_name']], on='station_name_id', how='left')
    demand = demand.merge(geo[['location_id', 'latitude', 'longitude']], on='location_id', how='left')
    demand['timestamp'] = pd.to_datetime(demand['timestamp'], utc=True)

    print(f"  Roh: {len(demand):,} Zeilen, {demand['station_id'].nunique()} Stationen")
    print(f"  Zeitraum: {demand['timestamp'].min().date()} bis {demand['timestamp'].max().date()}")

    print("\n  [1] Aggregation...")
    df = aggregate_station_level(demand, minutes=cfg.aggregation_minutes)
    print(f"      Shape: {df.shape}")

    print("  [2] Gap-Fill...")
    n_before = len(df)
    df = fill_missing_time_bins(df, minutes=cfg.aggregation_minutes)
    print(f"      {n_before:,} -> {len(df):,} (+{len(df)-n_before:,})")

    print("  [3] Stationen, Regime, Splits...")
    df, station_profile, train_end, val_end = prepare_stations_and_splits(df, cfg, city_name)

    print("  [4] Statistische Scoring-Pipeline...")
    df = run_statistical_pipeline(df, cfg, train_end)

    return df, station_profile, train_end, val_end

In [14]:
# ══════════════════════════════════════════════════════════════
# 11 — V17b Unified Evaluation (erweitert um Regime×Type Matrix)
# ══════════════════════════════════════════════════════════════
ANOM_TYPES = ["point_spike"]  # V20d: only spikes
REGIMES    = ["high", "mid", "low"]

def evaluate_v17b(
    meta: pd.DataFrame,
    score_col: str,
    experiment_name: str,
    val_start: pd.Timestamp,
    test_start: pd.Timestamp,
    verbose: bool = True
) -> Dict:
    meta = meta.copy()
    meta["split_eval"] = np.where(
        meta["hour_ts"] < val_start, "train",
        np.where(meta["hour_ts"] < test_start, "val", "test")
    )

    val_m  = meta[meta["split_eval"] == "val"].copy()
    test_m = meta[meta["split_eval"] == "test"].copy()

    if len(val_m) == 0 or len(test_m) == 0:
        print(f"  [{experiment_name}] WARNUNG: Keine VAL/TEST-Daten!")
        return {}

    results = {"experiment": experiment_name, "n_val": len(val_m), "n_test": len(test_m)}

    # --- Anomaly Rates ---
    n_test_total = len(test_m)
    n_test_anom  = (test_m["synth_label"] == 1).sum()
    results["anomaly_rate_total"] = n_test_anom / n_test_total

    for atype in ANOM_TYPES:
        n_t = (test_m["synth_type"] == atype).sum()
        results[f"rate_{atype}"] = n_t / n_test_total
        results[f"n_{atype}"] = int(n_t)

    # --- VAL: Threshold bestimmen ---
    y_val = val_m["synth_label"].astype(int).values
    s_val = val_m[score_col].values

    threshold = None
    if len(np.unique(y_val)) > 1:
        prec_v, rec_v, thr_v = precision_recall_curve(y_val, s_val)
        results["val_pr_auc"] = average_precision_score(y_val, s_val)
        if len(thr_v) > 0:
            f1_v = 2 * prec_v[:-1] * rec_v[:-1] / (prec_v[:-1] + rec_v[:-1] + 1e-12)
            best_idx = np.argmax(f1_v)
            threshold = float(thr_v[best_idx])
            results["val_best_f1"] = float(f1_v[best_idx])
            results["val_threshold"] = threshold

    if threshold is None:
        val_norm = val_m[val_m["synth_label"] == 0][score_col]
        if len(val_norm) > 0:
            threshold = float(val_norm.quantile(0.99))
        else:
            threshold = float(test_m[score_col].quantile(0.99))
        results["val_threshold"] = threshold
        results["threshold_source"] = "fallback_99pct"

    # --- TEST: Global Metriken ---
    y_test = test_m["synth_label"].astype(int).values
    s_test = test_m[score_col].values
    p_test = (s_test >= threshold).astype(int)

    if len(np.unique(y_test)) > 1:
        results["test_pr_auc"]  = average_precision_score(y_test, s_test)
        results["test_roc_auc"] = roc_auc_score(y_test, s_test)
        results["test_f1"]      = f1_score(y_test, p_test, zero_division=0)
        results["test_prec"]    = precision_score(y_test, p_test, zero_division=0)
        results["test_recall"]  = recall_score(y_test, p_test, zero_division=0)

    # --- TEST: Per-Type ---
    for atype in ANOM_TYPES:
        type_mask = test_m["synth_type"] == atype
        normal_mask = test_m["synth_label"] == 0
        sub = test_m[type_mask | normal_mask].copy()
        n_anom = int(type_mask.sum())

        if n_anom > 0 and len(sub) > 0:
            y_t = (sub["synth_type"] == atype).astype(int).values
            y_s = sub[score_col].values
            if len(set(y_t)) > 1:
                results[f"pr_{atype}"] = average_precision_score(y_t, y_s)
            type_seqs = test_m[type_mask]
            detected = (type_seqs[score_col] >= threshold).sum()
            results[f"dr_{atype}"] = detected / len(type_seqs) if len(type_seqs) > 0 else 0.0
        else:
            results[f"pr_{atype}"] = None
            results[f"dr_{atype}"] = None

    # --- TEST: Per-Type x Per-Regime (3x3 Matrix) ---
    regime_rows = []
    for atype in ANOM_TYPES:
        for regime in REGIMES:
            rm = test_m["demand_regime"] == regime
            tm = test_m["synth_type"] == atype
            nm = test_m["synth_label"] == 0
            n_a = int((rm & tm).sum())
            n_n = int((rm & nm).sum())
            pr, f1_val, dr = None, None, None
            if n_a > 0 and n_n > 0:
                sub = test_m[(rm & tm) | (rm & nm)]
                y_t = (sub["synth_type"] == atype).astype(int).values
                y_s = sub[score_col].values
                pr = average_precision_score(y_t, y_s) if len(set(y_t)) > 1 else None
                p_sub = (y_s >= threshold).astype(int)
                f1_val = f1_score(y_t, p_sub, zero_division=0)
                detected = (test_m[rm & tm][score_col] >= threshold).sum()
                dr = detected / n_a
            regime_rows.append({
                "type": atype, "regime": regime,
                "n_anom": n_a, "pr_auc": pr, "f1": f1_val, "dr": dr
            })
    results["regime_detail"] = pd.DataFrame(regime_rows)

    if verbose:
        print(f"\n  [{experiment_name}]")
        print(f"    Anomaly Rate: {results['anomaly_rate_total']:.4f} ({results['anomaly_rate_total']*100:.2f}%)")
        for at in ANOM_TYPES:
            print(f"      {at:15s}: n={results[f'n_{at}']:5d}, rate={results[f'rate_{at}']:.4f}")
        print(f"    Threshold: {threshold:.6f}")
        print(f"    TEST PR-AUC={results.get('test_pr_auc', 'N/A'):.4f}, "
              f"F1={results.get('test_f1', 'N/A'):.4f}, "
              f"Prec={results.get('test_prec', 'N/A'):.4f}, "
              f"Recall={results.get('test_recall', 'N/A'):.4f}")
        print(f"    Per-Type PR-AUC:")
        for at in ANOM_TYPES:
            v = results.get(f'pr_{at}')
            print(f"      {at:15s}: {v:.4f}" if v is not None else f"      {at:15s}: N/A")
        print(f"\n    Type × Regime Matrix (PR-AUC / F1):")
        rd = results["regime_detail"]
        for regime in REGIMES:
            vals = []
            for atype in ANOM_TYPES:
                row = rd[(rd["type"] == atype) & (rd["regime"] == regime)].iloc[0]
                pr_s = f"{row['pr_auc']:.3f}" if row['pr_auc'] is not None else "  N/A"
                f1_s = f"{row['f1']:.3f}" if row['f1'] is not None else "  N/A"
                vals.append(f"{pr_s}/{f1_s}")
            print(f"      {regime:4s}: " + "  |  ".join(vals))
        print(f"            {'spike':>11s}  |  {'drop':>11s}  |  {'collective':>11s}")

    return results

---
## Schritt 1: Städte laden

In [ ]:
# ══════════════════════════════════════════════════════════════
# 12 — Mannheim (Target) laden
# ══════════════════════════════════════════════════════════════
df_ma, profile_ma, train_end_ma, val_end_ma = prepare_city_data(
    CITY_DEMAND["Mannheim"], geo_path, station_names_path, cfg, "Mannheim"
)

print(f"\n{'='*70}")
print("  ZUSAMMENFASSUNG")
print(f"{'='*70}")
print(f"  MA: {df_ma.shape}, Stationen: {df_ma['station_id'].nunique()}")


  DATEN-PIPELINE: Mannheim
  Roh: 2,547,242 Zeilen, 123 Stationen
  Zeitraum: 2023-03-31 bis 2026-02-02

  [1] Aggregation...
      Shape: (1036182, 22)
  [2] Gap-Fill...
      1,036,182 -> 2,297,523 (+1,261,341)
  [3] Stationen, Regime, Splits...
  Aktive Stationen (Mannheim): 88
  Regime: {'mid': 31, 'high': 31, 'low': 31}
  Zeitraum: 2023-03-31 bis 2026-02-02 (1040 Tage)
  Split: Train < 2025-02-25, Val < 2025-07-30, Test ab 2025-07-30
  [4] Statistische Scoring-Pipeline...
  [1/5] Kontext-Keys...
  [2/5] Rolling z-Scores...
  [3/5] Poisson-Tail-Scores...
  [4/5] ECDF-Kalibrierung...
  [5/5] Labels...
  Label-Verteilung:
    {'normal': 2122687, 'grauzone_low': 46485, 'grauzone_high': 32007, 'anomal_high': 5640, 'anomal_low': 631}
    Anomalie-Rate: 0.0028 (0.28%)

  ZUSAMMENFASSUNG
  MA: (2207450, 70), Stationen: 88


---
## Schritt 2: Injection (V17b — ohne Contextuals)

In [ ]:
# ══════════════════════════════════════════════════════════════
# 13 — Injection: MA (ab train_end)
# ══════════════════════════════════════════════════════════════
df_ma_inj = inject_synthetic_anomalies_v17b(
    df_ma,
    inject_start=train_end_ma,
    injection_rate=cfg.injection_rate,
    seed=cfg.injection_seed + 1,
    verbose=True
)

vm = (df_ma_inj["hour_ts"] >= train_end_ma) & (df_ma_inj["hour_ts"] < val_end_ma)
tm = df_ma_inj["hour_ts"] >= val_end_ma
print(f"\nMA: VAL-Anomalien={df_ma_inj.loc[vm, 'synth_label'].sum():,}, "
      f"TEST-Anomalien={df_ma_inj.loc[tm, 'synth_label'].sum():,}")


  Synthetic Injection (V17b — ohne Contextuals):
    Inject-Zeilen gesamt:  748,728
    Injizierte Punkte:     11,230 (1.50%)

    Requested vs Actually Injected:
      point_spike      requested=11230  injected=11230

    Effective rates (of inject region):
      point_spike      1.500%

MA: VAL-Anomalien=5,484, TEST-Anomalien=5,746


---
## Schritt 3: Scaler + Sequenzen

In [22]:
# ══════════════════════════════════════════════════════════════
# 14 — Scaler + Sequenzen für BEIDE Städte
# ══════════════════════════════════════════════════════════════
ae_features = cfg.ae_features
score_features = cfg.ae_score_features
context_steps = cfg.ae_window_size - 1   # 23

def build_sequences_for_city(df_clean, df_inj, train_end, val_end, city_name, scaler_ext=None):
    """
    Build sequences for a city. If scaler_ext is provided, use it (transfer scenario).
    Returns: X_all, meta_all, scaler, X_train_normal, Y_train_normal, etc.
    """
    df_c = df_clean.copy().sort_values(["station_id", "hour_ts"]).reset_index(drop=True)
    df_c = df_c.dropna(subset=ae_features)
    train_mask_sc = df_c["hour_ts"] < train_end

    if scaler_ext is None:
        scaler = StandardScaler()
        scaler.fit(df_c.loc[train_mask_sc, ae_features])
        print(f"  [{city_name}] Scaler gefittet auf {train_mask_sc.sum():,} Train-Zeilen")
    else:
        scaler = scaler_ext
        print(f"  [{city_name}] Externer Scaler verwendet (Transfer)")

    df_inj_s = df_inj.copy().sort_values(["station_id", "hour_ts"]).reset_index(drop=True)
    df_inj_s = df_inj_s.dropna(subset=ae_features)

    df_inj_scaled = df_inj_s.copy()
    df_inj_scaled[ae_features] = scaler.transform(df_inj_s[ae_features])

    for col in ["synth_label", "synth_type", "original_n_lends", "original_n_returns",
                "demand_regime", "label_eval"]:
        if col in df_inj_s.columns:
            df_inj_scaled[col] = df_inj_s[col].values

    X_all, meta_all = make_sequences_with_window_labels(
        df_inj_scaled,
        feature_cols=ae_features,
        window_size=cfg.ae_window_size,
        synth_label_col="synth_label",
        synth_type_col="synth_type",
        require_contiguous=cfg.require_contiguous,
        agg_minutes=cfg.aggregation_minutes
    )

    meta_all["split_eval"] = np.where(
        meta_all["hour_ts"] < train_end, "train",
        np.where(meta_all["hour_ts"] < val_end, "val", "test")
    )

    print(f"  [{city_name}] Sequenzen: {X_all.shape}")
    print(f"  [{city_name}] Split: {meta_all['split_eval'].value_counts().to_dict()}")

    # Train normal + Val normal
    train_normal = (meta_all["split_eval"] == "train") & (meta_all["label_eval"] == "normal")
    val_normal   = (meta_all["split_eval"] == "val") & (meta_all["synth_label"] == 0) & (meta_all["label_eval"] == "normal")

    X_train = X_all[train_normal.values]
    X_val_c = X_all[val_normal.values]

    print(f"  [{city_name}] X_train (normal): {X_train.shape}, X_val (normal): {X_val_c.shape}")

    return {
        "X_all": X_all, "meta_all": meta_all, "scaler": scaler,
        "X_train": X_train, "X_val_clean": X_val_c,
        "train_normal_mask": train_normal, "val_normal_mask": val_normal,
        "train_end": train_end, "val_end": val_end,
    }


print("\n--- Mannheim (Target) ---")
ma_data = build_sequences_for_city(df_ma, df_ma_inj, train_end_ma, val_end_ma, "MA")


--- Mannheim (Target) ---


NameError: name 'df_ma' is not defined

---
## Experiment 1: MA Oracle (Target-Only 100%)

In [ ]:

# ══════════════════════════════════════════════════════════════
# X — V20d-Modelle laden (nur aus V20d-Pfad)
# ══════════════════════════════════════════════════════════════
from tensorflow import keras
import os

def try_load_city_models(models_dir, city_name):
    city_key = city_name.lower()
    fc_path = os.path.join(models_dir, f"{city_key}_oracle_forecaster.keras")
    ae_path = os.path.join(models_dir, f"{city_key}_oracle_autoencoder.keras")

    fc_model = keras.models.load_model(fc_path, compile=False) if os.path.exists(fc_path) else None
    ae_model = keras.models.load_model(ae_path, compile=False) if os.path.exists(ae_path) else None

    print(f"[{city_name}] FC:", "geladen" if fc_model is not None else "nicht gefunden", "→", fc_path)
    print(f"[{city_name}] AE:", "geladen" if ae_model is not None else "nicht gefunden", "→", ae_path)
    return fc_model, ae_model, fc_path, ae_path

ma_fc_loaded, ma_ae_loaded, ma_fc_path, ma_ae_path = try_load_city_models(PRELOAD_MODELS_DIR, "MA")
hd_fc_loaded, hd_ae_loaded, hd_fc_path, hd_ae_path = try_load_city_models(PRELOAD_MODELS_DIR, "HD")


[MA] FC: nicht gefunden → /content/drive/MyDrive/BA_Colab/data/v20d_spike_only/models/ma_oracle_forecaster.keras
[MA] AE: nicht gefunden → /content/drive/MyDrive/BA_Colab/data/v20d_spike_only/models/ma_oracle_autoencoder.keras
[HD] FC: nicht gefunden → /content/drive/MyDrive/BA_Colab/data/v20d_spike_only/models/hd_oracle_forecaster.keras
[HD] AE: nicht gefunden → /content/drive/MyDrive/BA_Colab/data/v20d_spike_only/models/hd_oracle_autoencoder.keras


In [ ]:
# ══════════════════════════════════════════════════════════════
# 15 — MA Oracle: Train + Evaluate Forecaster + AE on full MA Train
# ══════════════════════════════════════════════════════════════

def evaluate_fixed_threshold_v17b(
    meta: pd.DataFrame,
    score_col: str,
    experiment_name: str,
    val_start: pd.Timestamp,
    test_start: pd.Timestamp,
    fixed_threshold: float,
    threshold_source: str = "external_fixed",
    verbose: bool = True
) -> Dict:
    meta = meta.copy()
    meta["split_eval"] = np.where(
        meta["hour_ts"] < val_start, "train",
        np.where(meta["hour_ts"] < test_start, "val", "test")
    )

    val_m  = meta[meta["split_eval"] == "val"].copy()
    test_m = meta[meta["split_eval"] == "test"].copy()

    results = {
        "experiment": experiment_name,
        "n_val": len(val_m),
        "n_test": len(test_m),
        "val_threshold": float(fixed_threshold),
        "threshold_source": threshold_source,
    }

    if len(test_m) == 0:
        print(f"  [{experiment_name}] WARNUNG: Keine TEST-Daten!")
        return results

    n_test_total = len(test_m)
    n_test_anom  = int((test_m["synth_label"] == 1).sum())
    results["anomaly_rate_total"] = n_test_anom / n_test_total if n_test_total > 0 else np.nan

    for atype in ANOM_TYPES:
        n_t = int((test_m["synth_type"] == atype).sum())
        results[f"rate_{atype}"] = n_t / n_test_total if n_test_total > 0 else np.nan
        results[f"n_{atype}"] = n_t

    y_test = test_m["synth_label"].astype(int).values
    s_test = test_m[score_col].values
    p_test = (s_test >= fixed_threshold).astype(int)

    if len(np.unique(y_test)) > 1:
        results["test_pr_auc"]  = average_precision_score(y_test, s_test)
        results["test_roc_auc"] = roc_auc_score(y_test, s_test)
        results["test_f1"]      = f1_score(y_test, p_test, zero_division=0)
        results["test_prec"]    = precision_score(y_test, p_test, zero_division=0)
        results["test_recall"]  = recall_score(y_test, p_test, zero_division=0)

    for atype in ANOM_TYPES:
        type_mask = test_m["synth_type"] == atype
        normal_mask = test_m["synth_label"] == 0
        sub = test_m[type_mask | normal_mask].copy()
        n_anom = int(type_mask.sum())

        if n_anom > 0 and len(sub) > 0:
            y_t = (sub["synth_type"] == atype).astype(int).values
            y_s = sub[score_col].values
            if len(set(y_t)) > 1:
                results[f"pr_{atype}"] = average_precision_score(y_t, y_s)
            detected = int((test_m.loc[type_mask, score_col] >= fixed_threshold).sum())
            results[f"dr_{atype}"] = detected / n_anom if n_anom > 0 else 0.0
        else:
            results[f"pr_{atype}"] = None
            results[f"dr_{atype}"] = None

    regime_rows = []
    for atype in ANOM_TYPES:
        for regime in REGIMES:
            rm = test_m["demand_regime"] == regime
            tm = test_m["synth_type"] == atype
            nm = test_m["synth_label"] == 0
            n_a = int((rm & tm).sum())
            n_n = int((rm & nm).sum())
            pr, f1_val, dr = None, None, None
            if n_a > 0 and n_n > 0:
                sub = test_m[(rm & tm) | (rm & nm)]
                y_t = (sub["synth_type"] == atype).astype(int).values
                y_s = sub[score_col].values
                pr = average_precision_score(y_t, y_s) if len(set(y_t)) > 1 else None
                p_sub = (y_s >= fixed_threshold).astype(int)
                f1_val = f1_score(y_t, p_sub, zero_division=0)
                detected = int((test_m.loc[rm & tm, score_col] >= fixed_threshold).sum())
                dr = detected / n_a if n_a > 0 else None
            regime_rows.append({
                "type": atype, "regime": regime,
                "n_anom": n_a, "pr_auc": pr, "f1": f1_val, "dr": dr
            })
    results["regime_detail"] = pd.DataFrame(regime_rows)

    if verbose:
        print(f"\n  [{experiment_name}]")
        print(f"    Fixed threshold: {fixed_threshold:.6f} ({threshold_source})")
        print(f"    TEST PR-AUC: {results.get('test_pr_auc')}")
        print(f"    TEST F1:     {results.get('test_f1')}")
        print(f"    TEST Recall: {results.get('test_recall')}")

    return results


def run_oracle_experiment(city_data, cfg, city_name, models_dir,
                          fc_model_preloaded=None, ae_model_preloaded=None):
    """Train or load both FC and AE on full city training data, then evaluate."""
    X_train = city_data["X_train"]
    X_val_c = city_data["X_val_clean"]
    X_all   = city_data["X_all"]
    meta    = city_data["meta_all"].copy()
    train_end = city_data["train_end"]
    val_end   = city_data["val_end"]

    score_idx = [ae_features.index(f) for f in score_features]
    results = {}

    # --- Forecaster ---
    if fc_model_preloaded is not None:
        print(f"\n  [{city_name}] Verwende vorgeladenen Forecaster.")
        fc_model = fc_model_preloaded
    else:
        print(f"\n  [{city_name}] Training Forecaster (Oracle)...")
        fc_model = build_lstm_forecaster(
            n_input_features=len(ae_features),
            n_output_features=len(score_features),
            context_steps=context_steps,
            latent_dim=cfg.ae_latent_dim,
            n_layers=2,
            dropout=cfg.ae_dropout,
        )

        X_fc_train = X_train[:, :context_steps, :]
        Y_fc_train = X_train[:, -1, :][:, score_idx]
        X_fc_val   = X_val_c[:, :context_steps, :]
        Y_fc_val   = X_val_c[:, -1, :][:, score_idx]

        fc_model, _ = train_model_generic(
            fc_model, X_fc_train, Y_fc_train, X_fc_val, Y_fc_val,
            epochs=cfg.ae_epochs, lr=cfg.ae_lr,
            batch_size=cfg.ae_batch_size, early_stop=cfg.ae_early_stop,
            verbose=0
        )
        fc_model.save(f"{models_dir}/{city_name.lower()}_oracle_forecaster.keras")
        print(f"  [{city_name}] Forecaster gespeichert.")

    X_fc_input = X_all[:, :context_steps, :]
    X_actual_last = X_all[:, -1, :]
    meta["score_fc_raw"] = compute_forecast_scores(
        fc_model, X_fc_input, X_actual_last,
        score_features, ae_features, mode="directional"
    )
    meta = znorm_score_by_hour_station(
        meta, "score_fc_raw", train_end, val_end,
        new_col="score_fc_znorm_hs"
    )
    results["FC_Oracle"] = evaluate_v17b(
        meta, "score_fc_znorm_hs", f"{city_name}_Oracle_FC",
        val_start=train_end, test_start=val_end
    )

    # --- Autoencoder ---
    if ae_model_preloaded is not None:
        print(f"\n  [{city_name}] Verwende vorgeladenen Autoencoder.")
        ae_model = ae_model_preloaded
    else:
        print(f"\n  [{city_name}] Training Autoencoder (Oracle)...")
        ae_model = build_lstm_autoencoder(
            n_input_features=len(ae_features),
            window_size=cfg.ae_window_size,
            latent_dim=cfg.ae_latent_dim,
            n_layers=cfg.ae_layers,
            dropout=cfg.ae_dropout,
            bidirectional=cfg.use_bidirectional,
        )

        ae_model, _ = train_model_generic(
            ae_model, X_train, X_train, X_val_c, X_val_c,
            epochs=cfg.ae_epochs, lr=cfg.ae_lr,
            batch_size=cfg.ae_batch_size, early_stop=cfg.ae_early_stop,
            verbose=0
        )
        ae_model.save(f"{models_dir}/{city_name.lower()}_oracle_autoencoder.keras")
        print(f"  [{city_name}] Autoencoder gespeichert.")

    X_hat = predict_in_batches(ae_model, X_all)
    meta["score_ae_raw"] = compute_reconstruction_scores(
        X_all, X_hat, ae_features, score_features, mode="last_step_mean"
    )
    meta = znorm_score_by_hour_station(
        meta, "score_ae_raw", train_end, val_end,
        new_col="score_ae_znorm_hs"
    )
    results["AE_Oracle"] = evaluate_v17b(
        meta, "score_ae_znorm_hs", f"{city_name}_Oracle_AE",
        val_start=train_end, test_start=val_end
    )

    return results, fc_model, ae_model, meta

# Run MA Oracle
oracle_results, ma_fc_oracle, ma_ae_oracle, ma_meta = run_oracle_experiment(
    ma_data, cfg, "MA", MODELS_DIR,
    fc_model_preloaded=ma_fc_loaded,
    ae_model_preloaded=ma_ae_loaded,
)



  [MA] Training Forecaster (Oracle)...

Epoch 20: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.

Epoch 33: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.

Epoch 45: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.
Restoring model weights from the end of the best epoch: 48.
  [MA] Forecaster gespeichert.

  [MA_Oracle_FC]
    Anomaly Rate: 0.0136 (1.36%)
      point_spike    : n= 4988, rate=0.0136
    Threshold: 3.384629
    TEST PR-AUC=0.4484, F1=0.5121, Prec=0.5056, Recall=0.5188
    Per-Type PR-AUC:
      point_spike    : 0.4484

    Type × Regime Matrix (PR-AUC / F1):
      high: 0.498/0.514
      mid : 0.515/0.544
      low : 0.371/0.470
                  spike  |         drop  |   collective

  [MA] Training Autoencoder (Oracle)...

Epoch 45: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
Restoring model weights from the end of the best epoch: 50.
  [MA] Autoencoder gespeichert.

  [MA_Oracle_AE]
    A

---
## Schritt 4: HD Source-Modelle trainieren

In [ ]:
# ══════════════════════════════════════════════════════════════
# 16 — HD Source Models: Train FC + AE on full HD Train
# ══════════════════════════════════════════════════════════════
print("\n--- HD Source Model Training ---")
hd_oracle_results, hd_fc_model, hd_ae_model, hd_meta = run_oracle_experiment(
    hd_data, cfg, "HD", MODELS_DIR,
    fc_model_preloaded=hd_fc_loaded,
    ae_model_preloaded=hd_ae_loaded,
)

hd_meta_copy = hd_meta.copy()
print("\nHD Source-Modelle trainiert/geladen und gespeichert.")



--- HD Source Model Training ---

  [HD] Training Forecaster (Oracle)...

Epoch 28: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.

Epoch 45: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.

Epoch 49: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.
Epoch 49: early stopping
Restoring model weights from the end of the best epoch: 41.
  [HD] Forecaster gespeichert.

  [HD_Oracle_FC]
    Anomaly Rate: 0.0128 (1.28%)
      point_spike    : n= 1245, rate=0.0063
      point_drop     : n=  149, rate=0.0008
      collective     : n= 1138, rate=0.0058
    Threshold: 4.018925
    TEST PR-AUC=0.3811, F1=0.4483, Prec=0.4587, Recall=0.4384
    Per-Type PR-AUC:
      point_spike    : 0.2721
      point_drop     : 0.0034
      collective     : 0.2704

    Type × Regime Matrix (PR-AUC / F1):
      high: 0.340/0.404  |  0.011/0.005  |  0.398/0.445
      mid : 0.293/0.366  |  0.002/0.000  |  0.293/0.370
      low : 0.233/0.344  |  nan/nan  |  0.1

#Moment

In [15]:
# ══════════════════════════════════════════════════════════════
# MOMENT — Setup
# ══════════════════════════════════════════════════════════════
!pip install -q git+https://github.com/moment-timeseries-foundation-model/moment.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [16]:
import torch
from momentfm import MOMENTPipeline

MOMENT_MODEL_NAME = "AutonLab/MOMENT-1-base"
MOMENT_BATCH_SIZE = 64

# MOMENT bekommt dieselbe Window-Size wie der Oracle (24)
# und dieselben Score-Features (n_lends, n_returns)
# Die Zeitfeatures geben wir MOMENT NICHT — es bekommt nur Demand,
# weil es ein Foundation Model ist, das Zeitfeatures nicht braucht/erwartet.
MOMENT_SCORE_FEATURES = cfg.ae_score_features  # ["n_lends", "n_returns"]

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
print("Model :", MOMENT_MODEL_NAME)
print("Score-Features:", MOMENT_SCORE_FEATURES)
print("Window:", cfg.ae_window_size)

Device: cuda
Model : AutonLab/MOMENT-1-base
Score-Features: ['n_lends', 'n_returns']
Window: 24


In [17]:
# ══════════════════════════════════════════════════════════════
# MOMENT — Model + Scoring Helpers
# ══════════════════════════════════════════════════════════════

def load_moment_model(model_name=MOMENT_MODEL_NAME, device=device):
    model = MOMENTPipeline.from_pretrained(
        model_name,
        model_kwargs={"task_name": "reconstruction"}
    )
    model.init()
    model = model.to(device)
    model.eval()
    return model


def moment_reconstruction_scores(model, X_all_9feat, score_feature_indices,
                                  batch_size=MOMENT_BATCH_SIZE, device=device):
    """
    Fairer Vergleich: Nutzt dieselben X_all Fenster wie der Oracle (9 Features, 24 Steps).
    Extrahiert nur die Score-Features (n_lends, n_returns), schickt jede einzeln
    als univariate Reihe durch MOMENT, und mittelt den Last-Step-MSE über Channels.

    So sieht MOMENT exakt dieselben Daten und dasselbe Test-Set wie AE/FC.

    Args:
        model: MOMENT model
        X_all_9feat: [N, 24, 9] — dieselben Fenster wie Oracle
        score_feature_indices: Indizes von n_lends, n_returns in ae_features
        batch_size: Batch-Größe
        device: cuda/cpu

    Returns:
        scores: [N] — Last-Step-MSE gemittelt über Channels
    """
    n = X_all_9feat.shape[0]
    n_channels = len(score_feature_indices)
    scores_sum = np.zeros(n, dtype=np.float64)

    with torch.no_grad():
        for ch_pos, feat_idx in enumerate(score_feature_indices):
            ch_scores = []
            for i in range(0, n, batch_size):
                # Extrahiere einen Demand-Channel aus dem 9-Feature-Fenster
                xb = X_all_9feat[i:i+batch_size, :, feat_idx]       # [B, T]
                xb_t = torch.tensor(
                    xb[:, None, :], dtype=torch.float32, device=device
                )  # [B, 1, T]
                input_mask = torch.ones(
                    (xb_t.shape[0], xb_t.shape[-1]),
                    dtype=torch.long, device=device
                )

                out = model(x_enc=xb_t, input_mask=input_mask)
                xhat = out.reconstruction.detach().cpu().numpy()[:, 0, :]  # [B, T]

                # Last-Step-MSE — identisch zum Oracle AE Scoring
                err_last = ((xb - xhat) ** 2)[:, -1]
                ch_scores.append(err_last.astype(np.float64))

            scores_sum += np.concatenate(ch_scores)

    return (scores_sum / n_channels).astype(np.float32)

In [ ]:
# ══════════════════════════════════════════════════════════════
# MOMENT — Zero-Shot on MA (gleiche Daten wie Oracle)
# ══════════════════════════════════════════════════════════════

# Score-Feature-Indizes in ae_features bestimmen
score_feat_idx = [ae_features.index(f) for f in MOMENT_SCORE_FEATURES]
print(f"Score-Features: {MOMENT_SCORE_FEATURES}")
print(f"Indizes in ae_features: {score_feat_idx}")

# Sanity Check: gleiche Daten wie Oracle
print(f"\nOracle X_all shape: {ma_data['X_all'].shape}")
print(f"Oracle meta  shape: {ma_data['meta_all'].shape}")
print(f"Test-Punkte Oracle: {(ma_data['meta_all']['split_eval'] == 'test').sum()}")

# --- Zero-Shot ---
moment_zs_model = load_moment_model()

# Nutze exakt ma_data["X_all"] — dieselben Fenster wie der Oracle
moment_zs_meta = ma_data["meta_all"].copy()
moment_zs_meta["score_moment_raw"] = moment_reconstruction_scores(
    moment_zs_model,
    ma_data["X_all"],             # <-- gleiche Daten wie Oracle
    score_feat_idx,
    batch_size=MOMENT_BATCH_SIZE,
    device=device
)

# Z-Norm identisch wie bei Oracle (Hour × Station)
moment_zs_meta = znorm_score_by_hour_station(
    moment_zs_meta,
    raw_score_col="score_moment_raw",
    val_start=train_end_ma,
    test_start=val_end_ma,
    new_col="score_moment_znorm_hs"
)

# Evaluation auf exakt demselben Test-Set
moment_zs_results = evaluate_v17b(
    moment_zs_meta,
    score_col="score_moment_znorm_hs",
    experiment_name="MA_MOMENT_ZeroShot",
    val_start=train_end_ma,
    test_start=val_end_ma
)

Score-Features: ['n_lends', 'n_returns']
Indizes in ae_features: [0, 1]

Oracle X_all shape: (1981139, 24, 9)
Oracle meta  shape: (1981139, 78)
Test-Punkte Oracle: 367992


config.json:   0%|          | 0.00/949 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/454M [00:00<?, ?B/s]


  [MA_MOMENT_ZeroShot]
    Anomaly Rate: 0.0136 (1.36%)
      point_spike    : n= 4988, rate=0.0136
    Threshold: 7.093250
    TEST PR-AUC=0.4711, F1=0.5038, Prec=0.5705, Recall=0.4511
    Per-Type PR-AUC:
      point_spike    : 0.4711

    Type × Regime Matrix (PR-AUC / F1):
      high: 0.427/0.435
      mid : 0.530/0.514
      low : 0.525/0.586
                  spike  |         drop  |   collective


In [ ]:
# ══════════════════════════════════════════════════════════════
# MOMENT — Fine-Tune Helpers
# ══════════════════════════════════════════════════════════════
from torch.utils.data import DataLoader, TensorDataset

MOMENT_FT_EPOCHS = 5
MOMENT_FT_BATCH_SIZE = 32
MOMENT_FT_LR = 1e-4
MOMENT_FT_HEAD_ONLY = True
MOMENT_FT_MAX_TRAIN_WINDOWS = None


def maybe_subsample(X, max_n=None, seed=42):
    if (max_n is None) or (len(X) <= max_n):
        return X
    rng = np.random.RandomState(seed)
    idx = rng.choice(len(X), size=max_n, replace=False)
    return X[idx]


def make_moment_loaders_from_oracle_data(X_train_9feat, X_val_9feat,
                                          score_feat_idx, batch_size):
    """
    Extrahiert Score-Features aus den 9-Feature-Oracle-Fenstern
    und baut DataLoader für MOMENT Fine-Tune.
    Shape: [N, T, 9] -> [N, T, 2] (nur n_lends, n_returns)
    """
    X_train_demand = X_train_9feat[:, :, score_feat_idx].astype(np.float32)
    X_val_demand   = X_val_9feat[:, :, score_feat_idx].astype(np.float32)

    train_ds = TensorDataset(torch.from_numpy(X_train_demand))
    val_ds   = TensorDataset(torch.from_numpy(X_val_demand))

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=False)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False, drop_last=False)

    return train_loader, val_loader


def set_trainable_head_only(model):
    for p in model.parameters():
        p.requires_grad = False
    for p in model.head.parameters():
        p.requires_grad = True


def set_trainable_all(model):
    for p in model.parameters():
        p.requires_grad = True


def fine_tune_moment_reconstruction(
    model,
    train_loader,
    val_loader,
    epochs=MOMENT_FT_EPOCHS,
    lr=MOMENT_FT_LR,
    head_only=MOMENT_FT_HEAD_ONLY,
    save_path=None,
):
    """
    Fine-Tune MOMENT auf Rekonstruktion OHNE Masking.
    Ziel: Modell lernt normales Demand-Pattern zu rekonstruieren,
    Anomalien erzeugen dann höheren Rekonstruktionsfehler.
    """
    if head_only:
        set_trainable_head_only(model)
        print("Fine-Tune Mode: HEAD ONLY")
    else:
        set_trainable_all(model)
        print("Fine-Tune Mode: FULL MODEL")

    optimizer = torch.optim.Adam(
        [p for p in model.parameters() if p.requires_grad], lr=lr
    )
    criterion = torch.nn.MSELoss()

    best_val = np.inf
    best_state = None
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        train_losses = []

        for (batch_x,) in train_loader:
            # batch_x: [B, T, C] -> pro Channel univariat [B*C, 1, T]
            B, T, C = batch_x.shape
            batch_x = batch_x.to(device)
            x_flat = batch_x.permute(0, 2, 1).reshape(B * C, 1, T)  # [B*C, 1, T]
            input_mask = torch.ones(
                (x_flat.shape[0], T), dtype=torch.long, device=device
            )

            # Kein Masking — reines Rekonstruktionstraining
            output = model(x_enc=x_flat, input_mask=input_mask)
            loss = criterion(output.reconstruction, x_flat)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []
        with torch.no_grad():
            for (batch_x,) in val_loader:
                B, T, C = batch_x.shape
                batch_x = batch_x.to(device)
                x_flat = batch_x.permute(0, 2, 1).reshape(B * C, 1, T)
                input_mask = torch.ones(
                    (x_flat.shape[0], T), dtype=torch.long, device=device
                )
                output = model(x_enc=x_flat, input_mask=input_mask)
                loss = criterion(output.reconstruction, x_flat)
                val_losses.append(loss.item())

        train_loss = float(np.mean(train_losses))
        val_loss   = float(np.mean(val_losses))
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})
        improved = " ✓" if val_loss < best_val else ""
        print(f"  Epoch {epoch:02d} | train={train_loss:.6f} | val={val_loss:.6f}{improved}")

        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone()
                          for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    if save_path is not None:
        torch.save(model.state_dict(), save_path)
        print(f"  Saved: {save_path}")

    return model, pd.DataFrame(history)

In [ ]:
# ══════════════════════════════════════════════════════════════
# MOMENT — Fine-Tune on MA train-normal, evaluate on MA
# ══════════════════════════════════════════════════════════════

# Nicht mehr auf x_All wie zero shot

# Training nutzt weiterhin Train-Normal-Fenster (die braucht MOMENT zum Lernen)
X_train_ft = maybe_subsample(ma_data["X_train"], MOMENT_FT_MAX_TRAIN_WINDOWS, seed=42)
X_val_ft   = ma_data["X_val_clean"]

print(f"FT train windows: {X_train_ft.shape}")
print(f"FT val windows  : {X_val_ft.shape}")

train_loader, val_loader = make_moment_loaders_from_oracle_data(
    X_train_ft, X_val_ft, score_feat_idx,
    batch_size=MOMENT_FT_BATCH_SIZE
)

moment_ft_model = load_moment_model()

moment_ft_model, moment_ft_history = fine_tune_moment_reconstruction(
    moment_ft_model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=MOMENT_FT_EPOCHS,
    lr=MOMENT_FT_LR,
    head_only=MOMENT_FT_HEAD_ONLY,
    save_path=f"{MODELS_DIR}/ma_moment_ft.pt"
)

# Scoring nur auf Val+Test (gleicher Subset wie Zero-Shot)
moment_ft_meta = meta_valtest.drop(
    columns=[c for c in meta_valtest.columns if "moment" in c],
    errors="ignore"
).copy()

moment_ft_meta["score_moment_ft_raw"] = moment_reconstruction_scores(
    moment_ft_model,
    X_valtest,
    score_feat_idx,
    batch_size=MOMENT_BATCH_SIZE,
    device=device
)

moment_ft_meta = znorm_score_by_hour_station(
    moment_ft_meta,
    raw_score_col="score_moment_ft_raw",
    val_start=train_end_ma,
    test_start=val_end_ma,
    new_col="score_moment_ft_znorm_hs"
)

moment_ft_results = evaluate_v17b(
    moment_ft_meta,
    score_col="score_moment_ft_znorm_hs",
    experiment_name="MA_MOMENT_FineTune",
    val_start=train_end_ma,
    test_start=val_end_ma
)

print("\n--- Fine-Tune History ---")
print(moment_ft_history.to_string(index=False))

In [ ]:
# ══════════════════════════════════════════════════════════════
# MOMENT — Vergleich (alle auf exakt demselben Test-Set)
# ══════════════════════════════════════════════════════════════
rows = []

if "oracle_results" in globals():
    if "FC_Oracle" in oracle_results:
        r = oracle_results["FC_Oracle"]
        rows.append({
            "Experiment": r["experiment"],
            "Paradigma": "Forecast",
            "Training": "100% MA Train",
            "PR-AUC": r.get("test_pr_auc"),
            "F1": r.get("test_f1"),
            "Precision": r.get("test_prec"),
            "Recall": r.get("test_recall"),
        })
    if "AE_Oracle" in oracle_results:
        r = oracle_results["AE_Oracle"]
        rows.append({
            "Experiment": r["experiment"],
            "Paradigma": "Reconstruction",
            "Training": "100% MA Train",
            "PR-AUC": r.get("test_pr_auc"),
            "F1": r.get("test_f1"),
            "Precision": r.get("test_prec"),
            "Recall": r.get("test_recall"),
        })

rows.append({
    "Experiment": moment_zs_results["experiment"],
    "Paradigma": "Reconstruction",
    "Training": "Zero-Shot (pretrained)",
    "PR-AUC": moment_zs_results.get("test_pr_auc"),
    "F1": moment_zs_results.get("test_f1"),
    "Precision": moment_zs_results.get("test_prec"),
    "Recall": moment_zs_results.get("test_recall"),
})

rows.append({
    "Experiment": moment_ft_results["experiment"],
    "Paradigma": "Reconstruction",
    "Training": "FT Head on MA Train",
    "PR-AUC": moment_ft_results.get("test_pr_auc"),
    "F1": moment_ft_results.get("test_f1"),
    "Precision": moment_ft_results.get("test_prec"),
    "Recall": moment_ft_results.get("test_recall"),
})

comparison_df = pd.DataFrame(rows).sort_values("PR-AUC", ascending=False).reset_index(drop=True)
print(comparison_df.to_string(index=True))

# Hinweis zur Interpretation
print("\n" + "="*70)
print("INTERPRETATION:")
print("  • FC-Oracle vs MOMENT: Unterschiedliches Paradigma (Forecast vs Reconstruction).")
print("    FC sieht die Anomalie NICHT im Input → fairer nur als obere Schranke.")
print("  • AE-Oracle vs MOMENT: Gleiches Paradigma (Reconstruction, Last-Step-MSE).")
print("    → Direkter, fairer Vergleich. Gleiche Daten, gleiche Evaluation.")
print("  • MOMENT ZS vs FT: Bringt domänenspezifisches Fine-Tuning etwas?")
print("="*70)

---
## Multi-City Oracle + MOMENT Loop (für Modellfixierung und spätere Gewichtung)

Dieser Block erweitert V20d um mehrere Zielstädte.  
Er speichert:

- **Summary-Metriken** je Stadt und Modell
- **Fenster-/Sample-Level Scores** je Stadt und Modell
- **Thresholds / Metadaten** für spätere demand-gewichtete Auswertungen

Der Block verändert **nicht** das Modell selbst, sondern macht die Auswertung über mehrere Städte reproduzierbar.


In [19]:

# ══════════════════════════════════════════════════════════════
# MC-1 — City Discovery + Multi-City Config
# ══════════════════════════════════════════════════════════════
import re
from pathlib import Path

MULTICITY_RESULTS_DIR = f"{RESULTS_DIR}/multicity"
MULTICITY_PRED_DIR    = f"{MULTICITY_RESULTS_DIR}/predictions"
MULTICITY_SUMMARY_DIR = f"{MULTICITY_RESULTS_DIR}/summary"
MULTICITY_MODEL_DIR   = f"{MULTICITY_RESULTS_DIR}/models"

os.makedirs(MULTICITY_RESULTS_DIR, exist_ok=True)
os.makedirs(MULTICITY_PRED_DIR, exist_ok=True)
os.makedirs(MULTICITY_SUMMARY_DIR, exist_ok=True)
os.makedirs(MULTICITY_MODEL_DIR, exist_ok=True)

def city_to_key(city_name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", city_name.lower()).strip("_")

def discover_city_demand_paths(cleaned_base: str):
    demand_root = Path(cleaned_base) / "demand"
    city_map = {}
    if not demand_root.exists():
        print(f"WARNUNG: demand_root nicht gefunden: {demand_root}")
        return city_map

    for city_dir in sorted(demand_root.iterdir()):
        if not city_dir.is_dir():
            continue
        p = city_dir / "demand_cleaned.parquet"
        if p.exists():
            city_map[city_dir.name] = str(p)
    return city_map

DISCOVERED_CITY_DEMAND = discover_city_demand_paths(CLEANED_BASE)

# bisherige CITY_DEMAND-Einträge beibehalten, aber um alle gefundenen Städte erweitern
CITY_DEMAND_ALL = {**DISCOVERED_CITY_DEMAND, **CITY_DEMAND}

print(f"Gefundene Städte: {len(CITY_DEMAND_ALL)}")
print(sorted(CITY_DEMAND_ALL.keys())[:20])

# HIER Zielstädte setzen:
TARGET_CITIES = [
    "Mannheim",
    "Heidelberg",
    "Bochum",
    "Braunschweig",
    "Dortmund",
    "Duisburg",
    "Essen",
    "Marburg",
    "Winsen (Luhe)",

]

TARGET_CITIES = [c for c in TARGET_CITIES if c in CITY_DEMAND_ALL]
print("\nTarget Cities:", TARGET_CITIES)


Gefundene Städte: 17
['Bochum', 'Braunschweig', 'Dortmund', 'Duisburg', 'Essen', 'Freiburg', 'Gießen', 'Heidelberg', 'Kaiserslautern', 'Kassel', 'Leverkusen', 'Ludwigshafen', 'Mannheim', 'Marburg', 'Offenburg', 'Wiesbaden', 'Winsen (Luhe)']

Target Cities: ['Mannheim', 'Heidelberg', 'Bochum', 'Braunschweig', 'Dortmund', 'Duisburg', 'Essen', 'Marburg', 'Winsen (Luhe)']


In [20]:

# ══════════════════════════════════════════════════════════════
# MC-2 — Speicherung für spätere Gewichtung / Re-Scoring
# ══════════════════════════════════════════════════════════════
def save_window_level_scores(meta: pd.DataFrame,
                             city_name: str,
                             model_name: str,
                             out_dir: str,
                             score_cols=None,
                             extra_cols=None):
    """
    Speichert Fenster-/Sample-Level Scores so, dass spätere demand-gewichtete
    Auswertungen OHNE erneutes Training möglich sind.
    """
    score_cols = score_cols or []
    extra_cols = extra_cols or []

    base_cols = [
        "station_id", "hour_ts", "split_eval", "label_eval",
        "synth_label", "synth_type", "demand_regime"
    ]
    cols = [c for c in base_cols + score_cols + extra_cols if c in meta.columns]

    out = meta[cols].copy()
    out["city"] = city_name
    out["city_key"] = city_to_key(city_name)
    out["model"] = model_name

    # nützlich für spätere station-basierte Gewichtung
    station_activity = (
        out.groupby("station_id")
           .agg(
               n_windows=("station_id", "size"),
               n_test_windows=("split_eval", lambda s: int((s == "test").sum())),
               n_test_anom=("synth_label", lambda s: int(((out.loc[s.index, "split_eval"] == "test") & (s == 1)).sum()))
           )
           .reset_index()
    )
    out = out.merge(station_activity, on="station_id", how="left")

    fpath = f"{out_dir}/{city_to_key(city_name)}__{model_name}__window_scores.parquet"
    out.to_parquet(fpath, index=False)
    print(f"Gespeichert: {fpath} | rows={len(out):,}")
    return fpath

def flatten_result_dict(d: dict, city_name: str, model_name: str):
    row = {}
    for k, v in d.items():
        if isinstance(v, pd.DataFrame):
            continue
        row[k] = v
    row["city"] = city_name
    row["city_key"] = city_to_key(city_name)
    row["model"] = model_name
    return row

def save_regime_detail_if_present(result_dict: dict, city_name: str, model_name: str, out_dir: str):
    if "regime_detail" in result_dict and isinstance(result_dict["regime_detail"], pd.DataFrame):
        df = result_dict["regime_detail"].copy()
        df["city"] = city_name
        df["city_key"] = city_to_key(city_name)
        df["model"] = model_name
        fpath = f"{out_dir}/{city_to_key(city_name)}__{model_name}__regime_detail.csv"
        df.to_csv(fpath, index=False)
        return fpath
    return None

def add_station_demand_weight(meta: pd.DataFrame):
    """
    Fügt eine einfache Stationsaktivität hinzu, damit später
    High-/Low-Demand-Gewichtungen direkt berechnet werden können.
    """
    meta = meta.copy()
    demand_col_candidates = ["original_n_lends", "n_lends"]
    demand_col = next((c for c in demand_col_candidates if c in meta.columns), None)

    if demand_col is None:
        # fallback: nur Test-Windows zählen
        stat = meta.groupby("station_id").size().rename("station_activity_raw").reset_index()
    else:
        stat = (
            meta.groupby("station_id")[demand_col]
                .mean()
                .rename("station_activity_raw")
                .reset_index()
        )

    stat["station_activity_log1p"] = np.log1p(stat["station_activity_raw"].clip(lower=0))
    q33 = stat["station_activity_raw"].quantile(0.33)
    q67 = stat["station_activity_raw"].quantile(0.67)
    stat["station_activity_group"] = np.where(
        stat["station_activity_raw"] <= q33, "low",
        np.where(stat["station_activity_raw"] >= q67, "high", "medium")
    )

    meta = meta.merge(stat, on="station_id", how="left")
    return meta


In [25]:
# ══════════════════════════════════════════════════════════════
# X — Scaler + Sequenzen pro Stadt (generisch für Multi-City)
# ══════════════════════════════════════════════════════════════
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

ae_features = cfg.ae_features
score_features = cfg.ae_score_features
context_steps = cfg.ae_window_size - 1   # z. B. 23

def build_sequences_for_city(
    df_clean,
    df_inj,
    train_end,
    val_end,
    city_name,
    scaler_ext=None
):
    """
    Generische Sequenz-Erstellung pro Stadt.

    Parameter
    ----------
    df_clean : pd.DataFrame
        Clean-Daten der Stadt.
    df_inj : pd.DataFrame
        Injizierte Daten der Stadt.
    train_end : pd.Timestamp
        Split-Grenze Train/Val.
    val_end : pd.Timestamp
        Split-Grenze Val/Test.
    city_name : str
        Name der Stadt für Logging.
    scaler_ext : sklearn-Scaler | None
        Optional externer Scaler, z. B. für Transfer.
        Wenn None, wird der Scaler auf den Train-Daten dieser Stadt gefittet.

    Returns
    -------
    dict mit:
        X_all, meta_all, scaler,
        X_train, X_val_clean,
        train_normal_mask, val_normal_mask,
        train_end, val_end
    """

    # ----------------------------------------------------------
    # 1) Daten vorbereiten
    # ----------------------------------------------------------
    df_c = (
        df_clean.copy()
        .sort_values(["station_id", "hour_ts"])
        .reset_index(drop=True)
    )

    df_inj_s = (
        df_inj.copy()
        .sort_values(["station_id", "hour_ts"])
        .reset_index(drop=True)
    )

    # ----------------------------------------------------------
    # 2) Zeitstempel harmonisieren: alles UTC tz-aware
    #    -> verhindert tz-naive vs tz-aware Vergleichsfehler
    # ----------------------------------------------------------
    df_c["hour_ts"] = pd.to_datetime(df_c["hour_ts"], utc=True)
    df_inj_s["hour_ts"] = pd.to_datetime(df_inj_s["hour_ts"], utc=True)
    train_end = pd.to_datetime(train_end, utc=True)
    val_end   = pd.to_datetime(val_end, utc=True)

    # ----------------------------------------------------------
    # 3) NA-Handling auf Feature-Spalten
    # ----------------------------------------------------------
    df_c = df_c.dropna(subset=ae_features)
    df_inj_s = df_inj_s.dropna(subset=ae_features)

    # ----------------------------------------------------------
    # 4) Scaler fitten oder extern übernehmen
    # ----------------------------------------------------------
    train_mask_sc = df_c["hour_ts"] < train_end

    if scaler_ext is None:
        scaler = StandardScaler()
        scaler.fit(df_c.loc[train_mask_sc, ae_features])
        print(f"  [{city_name}] Scaler gefittet auf {train_mask_sc.sum():,} Train-Zeilen")
    else:
        scaler = scaler_ext
        print(f"  [{city_name}] Externer Scaler verwendet")

    # ----------------------------------------------------------
    # 5) Injizierte Daten skalieren
    # ----------------------------------------------------------
    df_inj_scaled = df_inj_s.copy()
    df_inj_scaled.loc[:, ae_features] = scaler.transform(df_inj_s[ae_features])

    # Relevante Metadaten erhalten
    meta_cols_to_keep = [
        "synth_label",
        "synth_type",
        "original_n_lends",
        "original_n_returns",
        "demand_regime",
        "label_eval",
        "station_id",
        "hour_ts",
        "n_lends",
        "n_returns",
    ]
    for col in meta_cols_to_keep:
        if col in df_inj_s.columns:
            df_inj_scaled[col] = df_inj_s[col].values

    # ----------------------------------------------------------
    # 6) Sequenzen bauen
    # ----------------------------------------------------------
    X_all, meta_all = make_sequences_with_window_labels(
        df_inj_scaled,
        feature_cols=ae_features,
        window_size=cfg.ae_window_size,
        synth_label_col="synth_label",
        synth_type_col="synth_type",
        require_contiguous=cfg.require_contiguous,
        agg_minutes=cfg.aggregation_minutes
    )

    # Falls hour_ts aus der Sequenzfunktion zurückkommt, ebenfalls harmonisieren
    if "hour_ts" in meta_all.columns:
        meta_all["hour_ts"] = pd.to_datetime(meta_all["hour_ts"], utc=True)

    # ----------------------------------------------------------
    # 7) Eval-Splits setzen
    # ----------------------------------------------------------
    meta_all["split_eval"] = np.where(
        meta_all["hour_ts"] < train_end, "train",
        np.where(meta_all["hour_ts"] < val_end, "val", "test")
    )

    print(f"  [{city_name}] Sequenzen: {X_all.shape}")
    print(f"  [{city_name}] Split: {meta_all['split_eval'].value_counts().to_dict()}")

    # ----------------------------------------------------------
    # 8) Nur normale Fenster für Train / Val-Clean
    # ----------------------------------------------------------
    train_normal = (
        (meta_all["split_eval"] == "train") &
        (meta_all["label_eval"] == "normal")
    )

    val_normal = (
        (meta_all["split_eval"] == "val") &
        (meta_all["synth_label"] == 0) &
        (meta_all["label_eval"] == "normal")
    )

    X_train = X_all[train_normal.values]
    X_val_c = X_all[val_normal.values]

    print(f"  [{city_name}] X_train (normal): {X_train.shape}, X_val (normal): {X_val_c.shape}")

    return {
        "X_all": X_all,
        "meta_all": meta_all,
        "scaler": scaler,
        "X_train": X_train,
        "X_val_clean": X_val_c,
        "train_normal_mask": train_normal,
        "val_normal_mask": val_normal,
        "train_end": train_end,
        "val_end": val_end,
    }

In [31]:
# ══════════════════════════════════════════════════════════════
# MC-ORACLE — Exakte V20d-Funktionsbasis für Oracle/Multi-City
# ══════════════════════════════════════════════════════════════

ANOM_TYPES = ["point_spike"]  # V20d: only spikes
REGIMES    = ["high", "mid", "low"]

# -------------------------------------------------------------
# 6 — Model Builders (AE + Forecaster)
# -------------------------------------------------------------
def build_lstm_autoencoder(
    n_input_features: int,
    window_size: int,
    latent_dim: int = 32,
    n_layers: int = 2,
    dropout: float = 0.1,
    bidirectional: bool = False,
    n_output_features: Optional[int] = None,
) -> keras.Model:
    n_output_features = n_output_features or n_input_features
    inputs = keras.Input(shape=(window_size, n_input_features), name="encoder_input")
    x = inputs
    for i in range(n_layers):
        return_sequences = (i < n_layers - 1)
        lstm = layers.LSTM(
            latent_dim, return_sequences=return_sequences,
            dropout=dropout, name=f"encoder_lstm_{i+1}"
        )
        x = layers.Bidirectional(lstm, name=f"encoder_bi_{i+1}")(x) if bidirectional else lstm(x)
    latent = x
    x = layers.RepeatVector(window_size, name="repeat_vector")(latent)
    for i in range(n_layers):
        lstm = layers.LSTM(
            latent_dim, return_sequences=True,
            dropout=dropout, name=f"decoder_lstm_{i+1}"
        )
        x = layers.Bidirectional(lstm, name=f"decoder_bi_{i+1}")(x) if bidirectional else lstm(x)
    outputs = layers.TimeDistributed(
        layers.Dense(n_output_features), name="output_dense"
    )(x)
    return keras.Model(inputs, outputs, name="lstm_autoencoder")


def build_lstm_forecaster(
    n_input_features: int,
    n_output_features: int,
    context_steps: int = 23,
    latent_dim: int = 32,
    n_layers: int = 2,
    dropout: float = 0.1,
) -> keras.Model:
    inputs = keras.Input(shape=(context_steps, n_input_features), name="forecast_input")
    x = inputs
    for i in range(n_layers):
        return_sequences = (i < n_layers - 1)
        x = layers.LSTM(
            latent_dim, return_sequences=return_sequences,
            dropout=dropout, name=f"forecast_lstm_{i+1}"
        )(x)
    outputs = layers.Dense(n_output_features, name="forecast_output")(x)
    return keras.Model(inputs, outputs, name="lstm_forecaster")


def train_model_generic(model, X_train, Y_train, X_val, Y_val,
                        epochs, lr, batch_size, early_stop, verbose=1):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr, clipnorm=1.0),
        loss="mse"
    )
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=early_stop,
            restore_best_weights=True, verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=4, verbose=1
        )
    ]
    history = model.fit(
        X_train, Y_train,
        validation_data=(X_val, Y_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks,
        verbose=verbose
    )
    return model, history


# -------------------------------------------------------------
# 7 — Scoring Helpers (AE + Forecaster)
# -------------------------------------------------------------
def predict_in_batches(model, X, batch_size=2048):
    preds = []
    for i in range(0, len(X), batch_size):
        preds.append(model.predict(X[i:i + batch_size], verbose=0))
    return np.concatenate(preds, axis=0) if preds else np.empty((0, 0, 0), dtype=np.float32)


def compute_reconstruction_scores(X, X_hat, input_features, score_features,
                                  mode="last_step_mean"):
    score_idx = [input_features.index(c) for c in score_features]
    err = (X - X_hat) ** 2
    err_score = err[:, :, score_idx]
    if mode == "last_step_mean":
        return err_score[:, -1, :].mean(axis=1)
    elif mode == "window_mean":
        return err_score.mean(axis=(1, 2))
    elif mode == "max_over_features":
        return err_score[:, -1, :].max(axis=1)
    else:
        raise ValueError(f"Unbekannter mode: {mode}")


def compute_forecast_scores(model, X_input, X_actual_last_step, score_features,
                            ae_features, mode="directional"):
    X_pred = predict_in_batches(model, X_input, batch_size=2048)
    score_idx = [ae_features.index(f) for f in score_features]
    X_actual = X_actual_last_step[:, score_idx]
    err = (X_pred - X_actual) ** 2
    if mode == "mean":
        return err.mean(axis=1)
    elif mode == "max":
        return err.max(axis=1)
    elif mode == "directional":
        abs_err = np.abs(X_pred - X_actual)
        denom = np.maximum(np.maximum(np.abs(X_actual), np.abs(X_pred)), 1e-6)
        rel_err = abs_err / denom
        return 0.5 * abs_err.mean(axis=1) + 0.5 * rel_err.mean(axis=1)
    else:
        raise ValueError(f"Unbekannter mode: {mode}")


# -------------------------------------------------------------
# 8 — Z-Normalisierung (Hour + Hour×Station)
# -------------------------------------------------------------
def znorm_score_by_hour(meta_df, raw_score_col, val_start, test_start,
                        new_col="score_znorm_hour"):
    meta = meta_df.copy()
    val_normal = (
        (meta["hour_ts"] >= val_start) &
        (meta["hour_ts"] < test_start) &
        (meta["synth_label"] == 0)
    )
    hour_col = meta["hour_ts"].dt.hour
    mu_per_hour = {}
    std_per_hour = {}
    for h in range(24):
        vals = meta.loc[val_normal & (hour_col == h), raw_score_col].dropna()
        mu_per_hour[h]  = vals.mean() if len(vals) > 0 else 0.0
        std_per_hour[h] = vals.std()  if len(vals) > 1 else 1.0

    hours = hour_col.values
    raw   = meta[raw_score_col].values
    znorm = np.zeros_like(raw)
    for i in range(len(raw)):
        h = hours[i]
        znorm[i] = (raw[i] - mu_per_hour[h]) / max(std_per_hour[h], 1e-8)
    meta[new_col] = znorm
    return meta, mu_per_hour, std_per_hour


def znorm_score_by_hour_station(meta_df, raw_score_col, val_start, test_start,
                                new_col="score_znorm_hour_station"):
    meta = meta_df.copy()
    val_normal = (
        (meta["hour_ts"] >= val_start) &
        (meta["hour_ts"] < test_start) &
        (meta["synth_label"] == 0)
    )
    hour_col = meta["hour_ts"].dt.hour
    meta["_hour_tmp"] = hour_col

    stats = (
        meta[val_normal]
        .groupby(["station_id", "_hour_tmp"])[raw_score_col]
        .agg(["mean", "std", "count"])
        .reset_index()
    )
    global_stats = (
        meta[val_normal]
        .groupby("_hour_tmp")[raw_score_col]
        .agg(["mean", "std"])
        .reset_index()
        .rename(columns={"mean": "g_mean", "std": "g_std"})
    )
    stats = stats.merge(global_stats, on="_hour_tmp", how="left")
    stats["use_mean"] = np.where(stats["count"] >= 10, stats["mean"], stats["g_mean"])
    stats["use_std"]  = np.where(stats["count"] >= 10, stats["std"],  stats["g_std"])
    stats["use_std"]  = stats["use_std"].clip(lower=1e-8)

    meta = meta.merge(
        stats[["station_id", "_hour_tmp", "use_mean", "use_std"]],
        on=["station_id", "_hour_tmp"], how="left"
    )
    meta["use_mean"] = meta["use_mean"].fillna(0.0)
    meta["use_std"]  = meta["use_std"].fillna(1.0).clip(lower=1e-8)
    meta[new_col] = (meta[raw_score_col] - meta["use_mean"]) / meta["use_std"]
    meta = meta.drop(columns=["_hour_tmp", "use_mean", "use_std"])
    return meta


# -------------------------------------------------------------
# 11 — V17b Unified Evaluation
# -------------------------------------------------------------
def evaluate_v17b(
    meta: pd.DataFrame,
    score_col: str,
    experiment_name: str,
    val_start: pd.Timestamp,
    test_start: pd.Timestamp,
    verbose: bool = True
) -> Dict:
    meta = meta.copy()
    meta["split_eval"] = np.where(
        meta["hour_ts"] < val_start, "train",
        np.where(meta["hour_ts"] < test_start, "val", "test")
    )

    val_m  = meta[meta["split_eval"] == "val"].copy()
    test_m = meta[meta["split_eval"] == "test"].copy()

    if len(val_m) == 0 or len(test_m) == 0:
        print(f"  [{experiment_name}] WARNUNG: Keine VAL/TEST-Daten!")
        return {}

    results = {"experiment": experiment_name, "n_val": len(val_m), "n_test": len(test_m)}

    # --- Anomaly Rates ---
    n_test_total = len(test_m)
    n_test_anom  = (test_m["synth_label"] == 1).sum()
    results["anomaly_rate_total"] = n_test_anom / n_test_total

    for atype in ANOM_TYPES:
        n_t = (test_m["synth_type"] == atype).sum()
        results[f"rate_{atype}"] = n_t / n_test_total
        results[f"n_{atype}"] = int(n_t)

    # --- VAL: Threshold bestimmen ---
    y_val = val_m["synth_label"].astype(int).values
    s_val = val_m[score_col].values

    threshold = None
    if len(np.unique(y_val)) > 1:
        prec_v, rec_v, thr_v = precision_recall_curve(y_val, s_val)
        results["val_pr_auc"] = average_precision_score(y_val, s_val)
        if len(thr_v) > 0:
            f1_v = 2 * prec_v[:-1] * rec_v[:-1] / (prec_v[:-1] + rec_v[:-1] + 1e-12)
            best_idx = np.argmax(f1_v)
            threshold = float(thr_v[best_idx])
            results["val_best_f1"] = float(f1_v[best_idx])
            results["val_threshold"] = threshold

    if threshold is None:
        val_norm = val_m[val_m["synth_label"] == 0][score_col]
        if len(val_norm) > 0:
            threshold = float(val_norm.quantile(0.99))
        else:
            threshold = float(test_m[score_col].quantile(0.99))
        results["val_threshold"] = threshold
        results["threshold_source"] = "fallback_99pct"

    # --- TEST: Global Metriken ---
    y_test = test_m["synth_label"].astype(int).values
    s_test = test_m[score_col].values
    p_test = (s_test >= threshold).astype(int)

    if len(np.unique(y_test)) > 1:
        results["test_pr_auc"]  = average_precision_score(y_test, s_test)
        results["test_roc_auc"] = roc_auc_score(y_test, s_test)
        results["test_f1"]      = f1_score(y_test, p_test, zero_division=0)
        results["test_prec"]    = precision_score(y_test, p_test, zero_division=0)
        results["test_recall"]  = recall_score(y_test, p_test, zero_division=0)

    # --- TEST: Per-Type ---
    for atype in ANOM_TYPES:
        type_mask = test_m["synth_type"] == atype
        normal_mask = test_m["synth_label"] == 0
        sub = test_m[type_mask | normal_mask].copy()
        n_anom = int(type_mask.sum())

        if n_anom > 0 and len(sub) > 0:
            y_t = (sub["synth_type"] == atype).astype(int).values
            y_s = sub[score_col].values
            if len(set(y_t)) > 1:
                results[f"pr_{atype}"] = average_precision_score(y_t, y_s)
            type_seqs = test_m[type_mask]
            detected = (type_seqs[score_col] >= threshold).sum()
            results[f"dr_{atype}"] = detected / len(type_seqs) if len(type_seqs) > 0 else 0.0
        else:
            results[f"pr_{atype}"] = None
            results[f"dr_{atype}"] = None

    # --- TEST: Per-Type x Per-Regime ---
    regime_rows = []
    for atype in ANOM_TYPES:
        for regime in REGIMES:
            rm = test_m["demand_regime"] == regime
            tm = test_m["synth_type"] == atype
            nm = test_m["synth_label"] == 0
            n_a = int((rm & tm).sum())
            n_n = int((rm & nm).sum())
            pr, f1_val, dr = None, None, None
            if n_a > 0 and n_n > 0:
                sub = test_m[(rm & tm) | (rm & nm)]
                y_t = (sub["synth_type"] == atype).astype(int).values
                y_s = sub[score_col].values
                pr = average_precision_score(y_t, y_s) if len(set(y_t)) > 1 else None
                p_sub = (y_s >= threshold).astype(int)
                f1_val = f1_score(y_t, p_sub, zero_division=0)
                detected = (test_m[rm & tm][score_col] >= threshold).sum()
                dr = detected / n_a
            regime_rows.append({
                "type": atype, "regime": regime,
                "n_anom": n_a, "pr_auc": pr, "f1": f1_val, "dr": dr
            })
    results["regime_detail"] = pd.DataFrame(regime_rows)

    if verbose:
        print(f"\n  [{experiment_name}]")
        print(f"    Anomaly Rate: {results['anomaly_rate_total']:.4f} ({results['anomaly_rate_total']*100:.2f}%)")
        for at in ANOM_TYPES:
            print(f"      {at:15s}: n={results[f'n_{at}']:5d}, rate={results[f'rate_{at}']:.4f}")
        print(f"    Threshold: {threshold:.6f}")
        print(f"    TEST PR-AUC={results.get('test_pr_auc', 'N/A'):.4f}, "
              f"F1={results.get('test_f1', 'N/A'):.4f}, "
              f"Prec={results.get('test_prec', 'N/A'):.4f}, "
              f"Recall={results.get('test_recall', 'N/A'):.4f}")
        print(f"    Per-Type PR-AUC:")
        for at in ANOM_TYPES:
            v = results.get(f'pr_{at}')
            print(f"      {at:15s}: {v:.4f}" if v is not None else f"      {at:15s}: N/A")
        print(f"\n    Type × Regime Matrix (PR-AUC / F1):")
        rd = results["regime_detail"]
        for regime in REGIMES:
            vals = []
            for atype in ANOM_TYPES:
                row = rd[(rd["type"] == atype) & (rd["regime"] == regime)].iloc[0]
                pr_s = f"{row['pr_auc']:.3f}" if row['pr_auc'] is not None else "  N/A"
                f1_s = f"{row['f1']:.3f}" if row['f1'] is not None else "  N/A"
                vals.append(f"{pr_s}/{f1_s}")
            print(f"      {regime:4s}: " + "  |  ".join(vals))
        print(f"            {'spike':>11s}  |  {'drop':>11s}  |  {'collective':>11s}")

    return results


# -------------------------------------------------------------
# 15 — Oracle Experiment
# -------------------------------------------------------------
def run_oracle_experiment(city_data, cfg, city_name, models_dir,
                          fc_model_preloaded=None, ae_model_preloaded=None):
    """Train or load both FC and AE on full city training data, then evaluate."""
    X_train = city_data["X_train"]
    X_val_c = city_data["X_val_clean"]
    X_all   = city_data["X_all"]
    meta    = city_data["meta_all"].copy()
    train_end = city_data["train_end"]
    val_end   = city_data["val_end"]

    score_idx = [ae_features.index(f) for f in score_features]
    results = {}

    # --- Forecaster ---
    if fc_model_preloaded is not None:
        print(f"\n  [{city_name}] Verwende vorgeladenen Forecaster.")
        fc_model = fc_model_preloaded
    else:
        print(f"\n  [{city_name}] Training Forecaster (Oracle)...")
        fc_model = build_lstm_forecaster(
            n_input_features=len(ae_features),
            n_output_features=len(score_features),
            context_steps=context_steps,
            latent_dim=cfg.ae_latent_dim,
            n_layers=2,
            dropout=cfg.ae_dropout,
        )

        X_fc_train = X_train[:, :context_steps, :]
        Y_fc_train = X_train[:, -1, :][:, score_idx]
        X_fc_val   = X_val_c[:, :context_steps, :]
        Y_fc_val   = X_val_c[:, -1, :][:, score_idx]

        fc_model, _ = train_model_generic(
            fc_model, X_fc_train, Y_fc_train, X_fc_val, Y_fc_val,
            epochs=cfg.ae_epochs, lr=cfg.ae_lr,
            batch_size=cfg.ae_batch_size, early_stop=cfg.ae_early_stop,
            verbose=0
        )
        fc_model.save(f"{models_dir}/{city_name.lower()}_oracle_forecaster.keras")
        print(f"  [{city_name}] Forecaster gespeichert.")

    X_fc_input = X_all[:, :context_steps, :]
    X_actual_last = X_all[:, -1, :]
    meta["score_fc_raw"] = compute_forecast_scores(
        fc_model, X_fc_input, X_actual_last,
        score_features, ae_features, mode="directional"
    )
    meta = znorm_score_by_hour_station(
        meta, "score_fc_raw", train_end, val_end,
        new_col="score_fc_znorm_hs"
    )
    results["FC_Oracle"] = evaluate_v17b(
        meta, "score_fc_znorm_hs", f"{city_name}_Oracle_FC",
        val_start=train_end, test_start=val_end
    )

    # --- Autoencoder ---
    if ae_model_preloaded is not None:
        print(f"\n  [{city_name}] Verwende vorgeladenen Autoencoder.")
        ae_model = ae_model_preloaded
    else:
        print(f"\n  [{city_name}] Training Autoencoder (Oracle)...")
        ae_model = build_lstm_autoencoder(
            n_input_features=len(ae_features),
            window_size=cfg.ae_window_size,
            latent_dim=cfg.ae_latent_dim,
            n_layers=cfg.ae_layers,
            dropout=cfg.ae_dropout,
            bidirectional=cfg.use_bidirectional,
        )

        ae_model, _ = train_model_generic(
            ae_model, X_train, X_train, X_val_c, X_val_c,
            epochs=cfg.ae_epochs, lr=cfg.ae_lr,
            batch_size=cfg.ae_batch_size, early_stop=cfg.ae_early_stop,
            verbose=0
        )
        ae_model.save(f"{models_dir}/{city_name.lower()}_oracle_autoencoder.keras")
        print(f"  [{city_name}] Autoencoder gespeichert.")

    X_hat = predict_in_batches(ae_model, X_all)
    meta["score_ae_raw"] = compute_reconstruction_scores(
        X_all, X_hat, ae_features, score_features, mode="last_step_mean"
    )
    meta = znorm_score_by_hour_station(
        meta, "score_ae_raw", train_end, val_end,
        new_col="score_ae_znorm_hs"
    )
    results["AE_Oracle"] = evaluate_v17b(
        meta, "score_ae_znorm_hs", f"{city_name}_Oracle_AE",
        val_start=train_end, test_start=val_end
    )

    return results, fc_model, ae_model, meta

In [26]:
# ══════════════════════════════════════════════════════════════
# MC-3 — Mehrere Städte laden, injecten, Sequenzen bauen
# ══════════════════════════════════════════════════════════════
multi_city_data = {}
multi_city_frames = {}

for i, city_name in enumerate(TARGET_CITIES, start=1):
    print("\n" + "="*90)
    print(f"[{i}/{len(TARGET_CITIES)}] CITY PIPELINE: {city_name}")
    print("="*90)

    df_city, profile_city, train_end_city, val_end_city = prepare_city_data(
        CITY_DEMAND_ALL[city_name], geo_path, station_names_path, cfg, city_name
    )

    df_city_inj = inject_synthetic_anomalies_v17b(
        df_city,
        inject_start=train_end_city,
        injection_rate=cfg.injection_rate,
        seed=cfg.injection_seed + i,
        verbose=True
    )

    city_data = build_sequences_for_city(
        df_clean=df_city,
        df_inj=df_city_inj,
        train_end=train_end_city,
        val_end=val_end_city,
        city_name=city_name,
        scaler_ext=None
    )

    # Aktivitätsinfo für spätere Gewichtung
    city_data["meta_all"] = add_station_demand_weight(city_data["meta_all"])

    multi_city_data[city_name] = city_data
    multi_city_frames[city_name] = {
        "df_clean": df_city,
        "df_inj": df_city_inj,
        "profile": profile_city,
        "train_end": train_end_city,
        "val_end": val_end_city,
    }

print("\nFertig. Geladene Städte:", list(multi_city_data.keys()))


[1/9] CITY PIPELINE: Mannheim

  DATEN-PIPELINE: Mannheim
  Roh: 2,547,242 Zeilen, 123 Stationen
  Zeitraum: 2023-03-31 bis 2026-02-02

  [1] Aggregation...
      Shape: (1036182, 22)
  [2] Gap-Fill...
      1,036,182 -> 2,297,523 (+1,261,341)
  [3] Stationen, Regime, Splits...
  Aktive Stationen (Mannheim): 88
  Regime: {'mid': 31, 'high': 31, 'low': 31}
  Zeitraum: 2023-03-31 bis 2026-02-02 (1040 Tage)
  Split: Train < 2025-02-25, Val < 2025-07-30, Test ab 2025-07-30
  [4] Statistische Scoring-Pipeline...
  [1/5] Kontext-Keys...
  [2/5] Rolling z-Scores...
  [3/5] Poisson-Tail-Scores...
  [4/5] ECDF-Kalibrierung...
  [5/5] Labels...
  Label-Verteilung:
    {'normal': 2122687, 'grauzone_low': 46485, 'grauzone_high': 32007, 'anomal_high': 5640, 'anomal_low': 631}
    Anomalie-Rate: 0.0028 (0.28%)

  Synthetic Injection (V17b — ohne Contextuals):
    Inject-Zeilen gesamt:  748,728
    Injizierte Punkte:     11,230 (1.50%)

    Requested vs Actually Injected:
      point_spike      requ

In [28]:
# ══════════════════════════════════════════════════════════════
# MC-X — Oracle Experiment pro Stadt (generisch)
# ══════════════════════════════════════════════════════════════
import os
import json
import numpy as np
import pandas as pd
from tensorflow import keras

def run_oracle_experiment(
    city_data,
    cfg,
    city_name,
    models_dir,
    fc_model_preloaded=None,
    ae_model_preloaded=None,
    save_models=True,
    verbose=True
):
    """
    Führt das Oracle-Experiment für genau eine Stadt aus.

    Erwartet in city_data:
        - X_all
        - meta_all
        - X_train
        - X_val_clean

    Nutzt:
        - train_forecaster(...)
        - train_autoencoder(...)
        - score_forecaster_windows(...)
        - score_autoencoder_windows(...)
        - evaluate_predictions(...)   # falls vorhanden
        oder berechnet zumindest Rohscores + Labels
    """
    os.makedirs(models_dir, exist_ok=True)

    X_all = city_data["X_all"]
    meta_all = city_data["meta_all"].copy()
    X_train = city_data["X_train"]
    X_val_clean = city_data["X_val_clean"]

    # ----------------------------------------------------------
    # Split-Masken
    # ----------------------------------------------------------
    test_mask = meta_all["split_eval"] == "test"
    y_test = meta_all.loc[test_mask, "synth_label"].astype(int).values

    # ----------------------------------------------------------
    # Modelle laden oder trainieren
    # ----------------------------------------------------------
    fc_model = fc_model_preloaded
    ae_model = ae_model_preloaded

    fc_path = os.path.join(models_dir, f"{city_name.lower()}_oracle_forecaster.keras")
    ae_path = os.path.join(models_dir, f"{city_name.lower()}_oracle_autoencoder.keras")

    # ---------- Forecaster ----------
    if fc_model is None:
        if verbose:
            print(f"[{city_name}] Trainiere Oracle Forecaster ...")
        fc_model, fc_hist = train_forecaster(
            X_train=X_train,
            X_val=X_val_clean,
            cfg=cfg,
            city_name=city_name
        )
        if save_models:
            fc_model.save(fc_path)
            if verbose:
                print(f"[{city_name}] FC gespeichert → {fc_path}")
    else:
        fc_hist = None
        if verbose:
            print(f"[{city_name}] Oracle Forecaster geladen")

    # ---------- Autoencoder ----------
    if ae_model is None:
        if verbose:
            print(f"[{city_name}] Trainiere Oracle Autoencoder ...")
        ae_model, ae_hist = train_autoencoder(
            X_train=X_train,
            X_val=X_val_clean,
            cfg=cfg,
            city_name=city_name
        )
        if save_models:
            ae_model.save(ae_path)
            if verbose:
                print(f"[{city_name}] AE gespeichert → {ae_path}")
    else:
        ae_hist = None
        if verbose:
            print(f"[{city_name}] Oracle Autoencoder geladen")

    # ----------------------------------------------------------
    # Scoring auf allen Fenstern
    # ----------------------------------------------------------
    if verbose:
        print(f"[{city_name}] Berechne Scores ...")

    fc_scores_all = score_forecaster_windows(fc_model, X_all, cfg=cfg)
    ae_scores_all = score_autoencoder_windows(ae_model, X_all, cfg=cfg)

    # Nur Test
    fc_scores_test = np.asarray(fc_scores_all)[test_mask.values]
    ae_scores_test = np.asarray(ae_scores_all)[test_mask.values]

    meta_test = meta_all.loc[test_mask].copy()
    meta_test["y_true"] = y_test
    meta_test["fc_score"] = fc_scores_test
    meta_test["ae_score"] = ae_scores_test

    # ----------------------------------------------------------
    # Auswertung
    # ----------------------------------------------------------
    results = []

    # Falls es im Notebook schon eine Standard-Eval-Funktion gibt, nutze sie
    if "evaluate_predictions" in globals():
        try:
            fc_eval = evaluate_predictions(
                y_true=y_test,
                scores=fc_scores_test,
                model_name="forecaster",
                city_name=city_name
            )
            fc_row = {"city": city_name, "setting": "oracle", "model": "forecaster"}
            if isinstance(fc_eval, dict):
                fc_row.update(fc_eval)
            results.append(fc_row)
        except Exception as e:
            print(f"[{city_name}] Warnung FC-Eval: {e}")

        try:
            ae_eval = evaluate_predictions(
                y_true=y_test,
                scores=ae_scores_test,
                model_name="autoencoder",
                city_name=city_name
            )
            ae_row = {"city": city_name, "setting": "oracle", "model": "autoencoder"}
            if isinstance(ae_eval, dict):
                ae_row.update(ae_eval)
            results.append(ae_row)
        except Exception as e:
            print(f"[{city_name}] Warnung AE-Eval: {e}")

    # Fallback: wenigstens Rohsummary
    if len(results) == 0:
        results = [
            {
                "city": city_name,
                "setting": "oracle",
                "model": "forecaster",
                "n_test": int(test_mask.sum()),
                "n_anom_test": int(y_test.sum())
            },
            {
                "city": city_name,
                "setting": "oracle",
                "model": "autoencoder",
                "n_test": int(test_mask.sum()),
                "n_anom_test": int(y_test.sum())
            }
        ]

    results_df = pd.DataFrame(results)

    meta_bundle = {
        "meta_all": meta_all,
        "meta_test": meta_test,
        "fc_history": fc_hist.history if fc_hist is not None and hasattr(fc_hist, "history") else None,
        "ae_history": ae_hist.history if ae_hist is not None and hasattr(ae_hist, "history") else None,
        "fc_model_path": fc_path,
        "ae_model_path": ae_path,
    }

    return results_df, fc_model, ae_model, meta_bundle

In [32]:

# ══════════════════════════════════════════════════════════════
# MC-4 — Multi-City Oracle (FC + AE) mit Speicherung
# ══════════════════════════════════════════════════════════════
from tensorflow import keras

def try_load_city_models_full(models_dir, city_name):
    city_key = city_to_key(city_name)
    fc_path = os.path.join(models_dir, f"{city_key}_oracle_forecaster.keras")
    ae_path = os.path.join(models_dir, f"{city_key}_oracle_autoencoder.keras")

    fc_model = keras.models.load_model(fc_path, compile=False) if os.path.exists(fc_path) else None
    ae_model = keras.models.load_model(ae_path, compile=False) if os.path.exists(ae_path) else None

    print(f"[{city_name}] FC:", "geladen" if fc_model is not None else "nicht gefunden", "→", fc_path)
    print(f"[{city_name}] AE:", "geladen" if ae_model is not None else "nicht gefunden", "→", ae_path)
    return fc_model, ae_model, fc_path, ae_path

oracle_summary_rows = []
oracle_regime_paths = []
oracle_score_paths = []
oracle_meta_by_city = {}

for city_name, city_data in multi_city_data.items():
    print("\n" + "="*90)
    print(f"ORACLE LOOP: {city_name}")
    print("="*90)

    fc_loaded, ae_loaded, _, _ = try_load_city_models_full(MULTICITY_MODEL_DIR, city_name)

    oracle_results_city, fc_model_city, ae_model_city, meta_city = run_oracle_experiment(
        city_data, cfg, city_name, MULTICITY_MODEL_DIR,
        fc_model_preloaded=fc_loaded,
        ae_model_preloaded=ae_loaded,
    )

    meta_city = add_station_demand_weight(meta_city)
    oracle_meta_by_city[city_name] = meta_city.copy()

    # FC
    if "score_fc_znorm_hs" in meta_city.columns and "FC_Oracle" in oracle_results_city:
        oracle_summary_rows.append(flatten_result_dict(oracle_results_city["FC_Oracle"], city_name, "FC_Oracle"))
        oracle_score_paths.append(
            save_window_level_scores(
                meta_city, city_name, "FC_Oracle", MULTICITY_PRED_DIR,
                score_cols=["score_fc_raw", "score_fc_znorm_hs"],
                extra_cols=["station_activity_raw", "station_activity_log1p", "station_activity_group"]
            )
        )
        p = save_regime_detail_if_present(oracle_results_city["FC_Oracle"], city_name, "FC_Oracle", MULTICITY_SUMMARY_DIR)
        if p is not None:
            oracle_regime_paths.append(p)

    # AE
    if "score_ae_znorm_hs" in meta_city.columns and "AE_Oracle" in oracle_results_city:
        oracle_summary_rows.append(flatten_result_dict(oracle_results_city["AE_Oracle"], city_name, "AE_Oracle"))
        oracle_score_paths.append(
            save_window_level_scores(
                meta_city, city_name, "AE_Oracle", MULTICITY_PRED_DIR,
                score_cols=["score_ae_raw", "score_ae_znorm_hs"],
                extra_cols=["station_activity_raw", "station_activity_log1p", "station_activity_group"]
            )
        )
        p = save_regime_detail_if_present(oracle_results_city["AE_Oracle"], city_name, "AE_Oracle", MULTICITY_SUMMARY_DIR)
        if p is not None:
            oracle_regime_paths.append(p)

oracle_summary_df = pd.DataFrame(oracle_summary_rows)
oracle_summary_path = f"{MULTICITY_SUMMARY_DIR}/oracle_summary.csv"
oracle_summary_df.to_csv(oracle_summary_path, index=False)

print("\nOracle Summary:")
display(oracle_summary_df.sort_values(["model", "test_pr_auc"], ascending=[True, False]))
print(f"\nOracle Summary gespeichert unter: {oracle_summary_path}")



ORACLE LOOP: Mannheim
[Mannheim] FC: nicht gefunden → /content/drive/MyDrive/BA_Colab/data/v20d_spike_only/multicity/models/mannheim_oracle_forecaster.keras
[Mannheim] AE: nicht gefunden → /content/drive/MyDrive/BA_Colab/data/v20d_spike_only/multicity/models/mannheim_oracle_autoencoder.keras

  [Mannheim] Training Forecaster (Oracle)...

Epoch 36: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.

Epoch 49: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.
Restoring model weights from the end of the best epoch: 50.
  [Mannheim] Forecaster gespeichert.

  [Mannheim_Oracle_FC]
    Anomaly Rate: 0.0136 (1.36%)
      point_spike    : n= 4988, rate=0.0136
    Threshold: 4.511058
    TEST PR-AUC=0.4986, F1=0.5486, Prec=0.5551, Recall=0.5423
    Per-Type PR-AUC:
      point_spike    : 0.4986

    Type × Regime Matrix (PR-AUC / F1):
      high: 0.502/0.515
      mid : 0.553/0.566
      low : 0.511/0.571
                  spike  |         drop  |   collective


,experiment,n_val,n_test,anomaly_rate_total,rate_point_spike,n_point_spike,val_pr_auc,val_best_f1,val_threshold,test_pr_auc,test_roc_auc,test_f1,test_prec,test_recall,pr_point_spike,dr_point_spike,city,city_key,model
17,Winsen (Luhe)_Oracle_AE,59840,71768,0.015369,0.015369,1103,0.824354,0.729649,8.284116,0.651449,0.989313,0.655901,0.603659,0.718042,0.651449,0.718042,Winsen (Luhe),winsen_luhe,AE_Oracle
9,Dortmund_Oracle_AE,82368,98677,0.013124,0.013124,1295,0.875205,0.884963,6.516618,0.624054,0.978139,0.654063,0.573256,0.761390,0.624054,0.761390,Dortmund,dortmund,AE_Oracle
13,Essen_Oracle_AE,54912,65296,0.012895,0.012895,842,0.819326,0.844714,6.536080,0.617377,0.982973,0.659091,0.583181,0.757720,0.617377,0.757720,Essen,essen,AE_Oracle
7,Braunschweig_Oracle_AE,106353,126861,0.012187,0.012187,1546,0.804223,0.810026,7.280339,0.605102,0.984811,0.616468,0.585125,0.651358,0.605102,0.651358,Braunschweig,braunschweig,AE_Oracle
11,Duisburg_Oracle_AE,42240,50578,0.012535,0.012535,634,0.785034,0.800633,6.497209,0.521605,0.972658,0.570751,0.572108,0.569401,0.521605,0.569401,Duisburg,duisburg,AE_Oracle
5,Bochum_Oracle_AE,65472,78396,0.011965,0.011965,938,0.848812,0.871937,6.500254,0.512149,0.981480,0.619932,0.513287,0.782516,0.512149,0.782516,Bochum,bochum,AE_Oracle
15,Marburg_Oracle_AE,149600,179519,0.013737,0.013737,2466,0.583739,0.546440,7.017307,0.432330,0.958512,0.460194,0.573156,0.384428,0.432330,0.384428,Marburg,marburg,AE_Oracle
1,Mannheim_Oracle_AE,306680,367992,0.013555,0.013555,4988,0.589364,0.534661,6.747976,0.424727,0.962734,0.472506,0.528522,0.427225,0.424727,0.427225,Mannheim,mannheim,AE_Oracle
3,Heidelberg_Oracle_AE,169145,197782,0.014015,0.014015,2772,0.556038,0.503895,6.245953,0.366151,0.961288,0.443929,0.439533,0.448413,0.366151,0.448413,Heidelberg,heidelberg,AE_Oracle
16,Winsen (Luhe)_Oracle_FC,59840,71768,0.015369,0.015369,1103,0.815857,0.736842,6.057621,0.688936,0.988274,0.675630,0.629601,0.728921,0.688936,0.728921,Winsen (Luhe),winsen_luhe,FC_Oracle



Oracle Summary gespeichert unter: /content/drive/MyDrive/BA_Colab/data/v20d_spike_only/multicity/summary/oracle_summary.csv


In [35]:
# ══════════════════════════════════════════════════════════════
# ORACLE POST-HOC — Helper Functions (kompakt)
# Stationsliste + schmale gewichtete Metriken
# ══════════════════════════════════════════════════════════════
import os
import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_recall_curve
)

MODEL_SCORE_COLS = {
    "FC_Oracle": "score_fc_znorm_hs",
    "AE_Oracle": "score_ae_znorm_hs",
}

GROUP_WEIGHT_MAP = {
    "low": 1.0,
    "mid": 2.0,
    "medium": 2.0,
    "high": 3.0
}

MIN_STATION_ANOM = 1
MIN_STATION_NEG  = 1


# -------------------------------------------------------------
# kleine Helfer
# -------------------------------------------------------------
def _safe_ap(y_true, y_score, sample_weight=None):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score, dtype=float)
    if len(np.unique(y_true)) < 2:
        return np.nan
    return float(average_precision_score(y_true, y_score, sample_weight=sample_weight))


def _best_f1_from_scores(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score, dtype=float)

    if len(np.unique(y_true)) < 2:
        return np.nan, np.nan

    prec, rec, thr = precision_recall_curve(y_true, y_score)
    f1 = 2 * prec * rec / np.clip(prec + rec, 1e-12, None)

    # precision_recall_curve gibt einen Punkt mehr als thresholds zurück
    if len(thr) == 0:
        return np.nan, np.nan

    f1_thr = f1[:-1]
    best_idx = int(np.nanargmax(f1_thr))
    return float(f1_thr[best_idx]), float(thr[best_idx])


def _infer_station_name(station_df: pd.DataFrame) -> str:
    candidate_cols = [
        "station_name",
        "name",
        "station_title",
        "station_label",
        "station"
    ]
    for c in candidate_cols:
        if c in station_df.columns and station_df[c].notna().any():
            vals = station_df[c].dropna().astype(str)
            if len(vals) > 0:
                return vals.mode().iloc[0]
    return ""


def _infer_station_group(station_df: pd.DataFrame) -> str:
    for c in ["station_activity_group", "demand_regime", "regime"]:
        if c in station_df.columns and station_df[c].notna().any():
            vals = station_df[c].dropna().astype(str)
            if len(vals) > 0:
                v = vals.mode().iloc[0]
                if v == "medium":
                    return "mid"
                return v
    return "unknown"


def _infer_station_activity_raw(station_df: pd.DataFrame) -> float:
    if "station_activity_raw" in station_df.columns and station_df["station_activity_raw"].notna().any():
        return float(station_df["station_activity_raw"].dropna().iloc[0])

    # fallback über Demand-Spalten
    demand_candidates = [
        "original_total_demand", "total_demand",
        "original_n_lends", "n_lends"
    ]
    for c in demand_candidates:
        if c in station_df.columns and station_df[c].notna().any():
            return float(station_df[c].mean())

    return 1.0


# -------------------------------------------------------------
# 1) Stationsliste pro Netzwerk/Run
# -------------------------------------------------------------
def build_station_test_table(test_df: pd.DataFrame, score_col: str) -> pd.DataFrame:
    """
    Pro Station:
    - station_name
    - station_id
    - test_pr_auc
    - test_f1 (bestes F1 aus Testkurve, rein deskriptiv)
    """
    rows = []

    for station_id, g in test_df.groupby("station_id"):
        y = g["synth_label"].astype(int).values
        s = g[score_col].astype(float).values

        n_pos = int((y == 1).sum())
        n_neg = int((y == 0).sum())

        if n_pos < MIN_STATION_ANOM or n_neg < MIN_STATION_NEG:
            pr_auc = np.nan
            best_f1 = np.nan
        else:
            pr_auc = _safe_ap(y, s)
            best_f1, _ = _best_f1_from_scores(y, s)

        rows.append({
            "station_name": _infer_station_name(g),
            "station_id": station_id,
            "test_pr_auc": pr_auc,
            "test_f1": best_f1,
            "n_test_windows": len(g),
            "n_test_anom": n_pos,
            "station_group": _infer_station_group(g),
            "station_activity_raw": _infer_station_activity_raw(g),
        })

    out = pd.DataFrame(rows)

    if len(out) == 0:
        return out

    out["group_weight"] = out["station_group"].map(GROUP_WEIGHT_MAP).fillna(1.0)
    out["sqrt_activity_weight"] = np.sqrt(out["station_activity_raw"].clip(lower=0) + 1e-12)

    out = out.sort_values(
        ["test_pr_auc", "test_f1", "n_test_anom", "n_test_windows"],
        ascending=[False, False, False, False]
    ).reset_index(drop=True)

    return out[[
        "station_name",
        "station_id",
        "test_pr_auc",
        "test_f1",
        "n_test_windows",
        "n_test_anom",
        "station_group",
        "station_activity_raw",
        "group_weight",
        "sqrt_activity_weight"
    ]]


# -------------------------------------------------------------
# 2) Schmale Run-Metrik-Tabelle
# -------------------------------------------------------------
def compute_compact_weighted_metrics(
    test_df: pd.DataFrame,
    score_col: str,
    station_table: pd.DataFrame
) -> dict:
    """
    Kompakte Metriken pro Run:
    - micro_pr_auc
    - macro_station_pr_auc
    - weighted_macro_station_pr_auc_group
    - weighted_macro_station_pr_auc_sqrt_activity
    """
    y = test_df["synth_label"].astype(int).values
    s = test_df[score_col].astype(float).values

    valid_station_df = station_table.dropna(subset=["test_pr_auc"]).copy()

    out = {
        "n_test_windows": int(len(test_df)),
        "n_test_anom": int((y == 1).sum()),
        "n_valid_stations": int(len(valid_station_df)),
        "micro_pr_auc": _safe_ap(y, s),
    }

    if len(valid_station_df) > 0:
        pr = valid_station_df["test_pr_auc"].astype(float).values
        gw = valid_station_df["group_weight"].astype(float).values
        aw = valid_station_df["sqrt_activity_weight"].astype(float).values

        out["macro_station_pr_auc"] = float(np.mean(pr))
        out["weighted_macro_pr_auc_group"] = float(np.average(pr, weights=gw))
        out["weighted_macro_pr_auc_sqrt_activity"] = float(np.average(pr, weights=aw))
    else:
        out["macro_station_pr_auc"] = np.nan
        out["weighted_macro_pr_auc_group"] = np.nan
        out["weighted_macro_pr_auc_sqrt_activity"] = np.nan

    return out

In [36]:
# ══════════════════════════════════════════════════════════════
# ORACLE POST-HOC — Ausführung (kompakt)
# Stationslisten + schmale Metrik-Tabelle
# ══════════════════════════════════════════════════════════════

oracle_station_tables = {}   # key: (city_name, model_name) -> station_table
oracle_compact_rows = []

for city_name, meta_city in oracle_meta_by_city.items():
    print("\n" + "="*100)
    print(f"POST-HOC ORACLE ANALYSIS: {city_name}")
    print("="*100)

    test_df = meta_city.loc[meta_city["split_eval"] == "test"].copy()

    if len(test_df) == 0:
        print("Keine TEST-Daten gefunden, übersprungen.")
        continue

    for model_name, score_col in MODEL_SCORE_COLS.items():
        if score_col not in test_df.columns:
            print(f"[{city_name} | {model_name}] Score-Spalte fehlt: {score_col}")
            continue

        # 1) Stationsliste erzeugen
        station_table = build_station_test_table(test_df, score_col=score_col)
        oracle_station_tables[(city_name, model_name)] = station_table

        print(f"\n[{city_name} | {model_name}] Stationsliste (sortiert nach test_pr_auc)")
        display(
            station_table[[
                "station_name",
                "station_id",
                "test_pr_auc",
                "test_f1"
            ]]
        )

        # 2) Kompakte gewichtete Run-Metriken
        compact_metrics = compute_compact_weighted_metrics(
            test_df=test_df,
            score_col=score_col,
            station_table=station_table
        )

        compact_metrics.update({
            "city": city_name,
            "city_key": city_to_key(city_name) if "city_to_key" in globals() else city_name.lower().replace(" ", "_"),
            "model": model_name
        })

        oracle_compact_rows.append(compact_metrics)

# -------------------------------------------------------------
# Kompakte Ergebnis-Tabelle
# -------------------------------------------------------------
oracle_compact_metrics_df = pd.DataFrame(oracle_compact_rows)

desired_cols = [
    "city",
    "model",
    "n_valid_stations",
    "micro_pr_auc",
    "macro_station_pr_auc",
    "weighted_macro_pr_auc_group",
    "weighted_macro_pr_auc_sqrt_activity",
    "n_test_anom"
]
desired_cols = [c for c in desired_cols if c in oracle_compact_metrics_df.columns]

oracle_compact_metrics_df = oracle_compact_metrics_df[desired_cols].sort_values(
    ["model", "weighted_macro_pr_auc_sqrt_activity"],
    ascending=[True, False]
).reset_index(drop=True)

print("\n" + "="*100)
print("ORACLE — KOMPAKTE METRIK-TABELLE")
print("="*100)
display(oracle_compact_metrics_df)

# -------------------------------------------------------------
# Optional speichern
# -------------------------------------------------------------
oracle_station_dir = f"{MULTICITY_SUMMARY_DIR}/oracle_station_tables"
os.makedirs(oracle_station_dir, exist_ok=True)

for (city_name, model_name), df_ in oracle_station_tables.items():
    city_key = city_to_key(city_name) if "city_to_key" in globals() else city_name.lower().replace(" ", "_")
    out_path = f"{oracle_station_dir}/{city_key}__{model_name}__stations.csv"
    df_[["station_name", "station_id", "test_pr_auc", "test_f1"]].to_csv(out_path, index=False)

oracle_compact_metrics_path = f"{MULTICITY_SUMMARY_DIR}/oracle_compact_metrics.csv"
oracle_compact_metrics_df.to_csv(oracle_compact_metrics_path, index=False)

print(f"\nStationslisten gespeichert unter: {oracle_station_dir}")
print(f"Kompakte Metrik-Tabelle gespeichert unter: {oracle_compact_metrics_path}")


POST-HOC ORACLE ANALYSIS: Mannheim

[Mannheim | FC_Oracle] Stationsliste (sortiert nach test_pr_auc)


,station_name,station_id,test_pr_auc,test_f1
0,Turleyplatz,29794323,0.855396,0.851064
1,Seckenheim - Rathaus,378596121,0.811727,0.816327
2,Oststadt - Beethovenstraße,28455055,0.791221,0.719101
3,Oststadt - Augusta Anlage,195402059,0.786552,0.750000
4,Neckarstadt-West – Marchivum,30612240,0.731647,0.731034
...,...,...,...,...
77,Roche - Oppauer Straße,57585616,0.366441,0.535714
78,Lindenhof - Jugendherberge,121376,0.341677,0.468750
79,Lindenhof - Hanns-Glückstein-Park,28450242,0.316281,0.411765
80,Feudenheim - St. Peter & Paul Kirche,378595589,0.299289,0.437500



[Mannheim | AE_Oracle] Stationsliste (sortiert nach test_pr_auc)


,station_name,station_id,test_pr_auc,test_f1
0,Herzogenried - Sandgewann,24559076,0.870472,0.842105
1,Turleyplatz,29794323,0.774876,0.764706
2,Seckenheim - Rathaus,378596121,0.749533,0.740741
3,Oststadt - Augusta Anlage,195402059,0.698719,0.686567
4,Oststadt - Beethovenstraße,28455055,0.669334,0.680412
...,...,...,...,...
77,Franklin - George-Washington-Straße,38930150,0.291421,0.500000
78,Roche - Oppauer Straße,57585616,0.259652,0.440678
79,Lindenhof - Hanns-Glückstein-Park,28450242,0.240374,0.339181
80,DHBW Mannheim - Campus Coblitzallee,4726808,0.188082,0.333333



POST-HOC ORACLE ANALYSIS: Heidelberg

[Heidelberg | FC_Oracle] Stationsliste (sortiert nach test_pr_auc)


...


[Winsen (Luhe) | AE_Oracle] Stationsliste (sortiert nach test_pr_auc)


,station_name,station_id,test_pr_auc,test_f1
0,Kapelle St. Georg,87641918,0.959881,0.923077
1,Von-Somnitz-Ring,87642200,0.877119,0.805556
2,Albert-Schweitzer-Straße,162363388,0.858303,0.811429
3,Stadthalle,87642769,0.831282,0.813559
4,Schlossplatz / Rathaus,87642371,0.809912,0.757576
5,Deichstr.,87642883,0.808184,0.764912
6,Eckermannstr.,87640674,0.790193,0.748718
7,Bleiche,149877579,0.754092,0.770270
8,An der Kleinbahn,149876031,0.680232,0.645418
9,Die Insel,87642602,0.671268,0.688000



ORACLE — KOMPAKTE METRIK-TABELLE


,city,model,n_valid_stations,micro_pr_auc,macro_station_pr_auc,weighted_macro_pr_auc_group,weighted_macro_pr_auc_sqrt_activity,n_test_anom
0,Essen,AE_Oracle,52,0.617377,0.733436,0.731956,0.729984,842
1,Dortmund,AE_Oracle,76,0.624054,0.729490,0.723881,0.719298,1295
2,Winsen (Luhe),AE_Oracle,16,0.651449,0.694824,0.703037,0.689101,1103
3,Braunschweig,AE_Oracle,80,0.605102,0.695048,0.683774,0.685333,1546
4,Duisburg,AE_Oracle,38,0.521605,0.677744,0.679875,0.683678,634
5,Bochum,AE_Oracle,62,0.512149,0.627268,0.625340,0.628507,938
6,Marburg,AE_Oracle,40,0.432330,0.478846,0.478522,0.484752,2466
7,Mannheim,AE_Oracle,82,0.424727,0.480244,0.472130,0.474285,4988
8,Heidelberg,AE_Oracle,45,0.366151,0.413393,0.415235,0.412429,2772
9,Essen,FC_Oracle,52,0.651303,0.771060,0.769867,0.767844,842



Stationslisten gespeichert unter: /content/drive/MyDrive/BA_Colab/data/v20d_spike_only/multicity/summary/oracle_station_tables
Kompakte Metrik-Tabelle gespeichert unter: /content/drive/MyDrive/BA_Colab/data/v20d_spike_only/multicity/summary/oracle_compact_metrics.csv
Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


In [ ]:

# ══════════════════════════════════════════════════════════════
# MC-5 — Multi-City MOMENT (Zero-Shot + Fine-Tune) mit Speicherung
# ══════════════════════════════════════════════════════════════
score_feat_idx = [ae_features.index(f) for f in MOMENT_SCORE_FEATURES]
moment_summary_rows = []
moment_histories = {}
moment_score_paths = []

def run_moment_city(city_name: str, city_data: dict, model_dir: str):
    train_end = city_data["train_end"]
    val_end   = city_data["val_end"]

    # ----- Zero-Shot -----
    print(f"\n[{city_name}] MOMENT Zero-Shot")
    zs_model = load_moment_model()
    zs_meta = city_data["meta_all"].copy()
    zs_meta["score_moment_raw"] = moment_reconstruction_scores(
        zs_model,
        city_data["X_all"],
        score_feat_idx,
        batch_size=MOMENT_BATCH_SIZE,
        device=device
    )
    zs_meta = znorm_score_by_hour_station(
        zs_meta,
        raw_score_col="score_moment_raw",
        val_start=train_end,
        test_start=val_end,
        new_col="score_moment_znorm_hs"
    )
    zs_meta = add_station_demand_weight(zs_meta)
    zs_res = evaluate_v17b(
        zs_meta,
        score_col="score_moment_znorm_hs",
        experiment_name=f"{city_name}_MOMENT_ZeroShot",
        val_start=train_end,
        test_start=val_end
    )

    # ----- Fine-Tune -----
    print(f"\n[{city_name}] MOMENT Fine-Tune")
    X_train_ft = maybe_subsample(city_data["X_train"], MOMENT_FT_MAX_TRAIN_WINDOWS, seed=42)
    X_val_ft   = city_data["X_val_clean"]

    train_loader, val_loader = make_moment_loaders_from_oracle_data(
        X_train_ft, X_val_ft, score_feat_idx,
        batch_size=MOMENT_FT_BATCH_SIZE
    )

    ft_model = load_moment_model()
    city_key = city_to_key(city_name)
    ft_model, ft_history = fine_tune_moment_reconstruction(
        ft_model,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=MOMENT_FT_EPOCHS,
        lr=MOMENT_FT_LR,
        head_only=MOMENT_FT_HEAD_ONLY,
        save_path=f"{model_dir}/{city_key}_moment_ft.pt"
    )

    mask_valtest = city_data["meta_all"]["split_eval"].isin(["val", "test"]).values
    X_valtest = city_data["X_all"][mask_valtest]
    meta_valtest = city_data["meta_all"].loc[mask_valtest].copy()

    ft_meta = meta_valtest.copy()
    ft_meta["score_moment_ft_raw"] = moment_reconstruction_scores(
        ft_model,
        X_valtest,
        score_feat_idx,
        batch_size=MOMENT_BATCH_SIZE,
        device=device
    )
    ft_meta = znorm_score_by_hour_station(
        ft_meta,
        raw_score_col="score_moment_ft_raw",
        val_start=train_end,
        test_start=val_end,
        new_col="score_moment_ft_znorm_hs"
    )
    ft_meta = add_station_demand_weight(ft_meta)
    ft_res = evaluate_v17b(
        ft_meta,
        score_col="score_moment_ft_znorm_hs",
        experiment_name=f"{city_name}_MOMENT_FineTune",
        val_start=train_end,
        test_start=val_end
    )

    return zs_res, zs_meta, ft_res, ft_meta, ft_history

for city_name, city_data in multi_city_data.items():
    print("\n" + "="*90)
    print(f"MOMENT LOOP: {city_name}")
    print("="*90)

    zs_res, zs_meta, ft_res, ft_meta, ft_history = run_moment_city(
        city_name, city_data, MULTICITY_MODEL_DIR
    )
    moment_histories[city_name] = ft_history

    moment_summary_rows.append(flatten_result_dict(zs_res, city_name, "MOMENT_ZeroShot"))
    moment_summary_rows.append(flatten_result_dict(ft_res, city_name, "MOMENT_FineTune"))

    moment_score_paths.append(
        save_window_level_scores(
            zs_meta, city_name, "MOMENT_ZeroShot", MULTICITY_PRED_DIR,
            score_cols=["score_moment_raw", "score_moment_znorm_hs"],
            extra_cols=["station_activity_raw", "station_activity_log1p", "station_activity_group"]
        )
    )
    moment_score_paths.append(
        save_window_level_scores(
            ft_meta, city_name, "MOMENT_FineTune", MULTICITY_PRED_DIR,
            score_cols=["score_moment_ft_raw", "score_moment_ft_znorm_hs"],
            extra_cols=["station_activity_raw", "station_activity_log1p", "station_activity_group"]
        )
    )

    p = save_regime_detail_if_present(zs_res, city_name, "MOMENT_ZeroShot", MULTICITY_SUMMARY_DIR)
    if p is not None:
        oracle_regime_paths.append(p)
    p = save_regime_detail_if_present(ft_res, city_name, "MOMENT_FineTune", MULTICITY_SUMMARY_DIR)
    if p is not None:
        oracle_regime_paths.append(p)

moment_summary_df = pd.DataFrame(moment_summary_rows)
moment_summary_path = f"{MULTICITY_SUMMARY_DIR}/moment_summary.csv"
moment_summary_df.to_csv(moment_summary_path, index=False)

print("\nMOMENT Summary:")
display(moment_summary_df.sort_values(["model", "test_pr_auc"], ascending=[True, False]))
print(f"\nMOMENT Summary gespeichert unter: {moment_summary_path}")


In [ ]:

# ══════════════════════════════════════════════════════════════
# MC-6 — Gesamtvergleich + Check für spätere Gewichtung
# ══════════════════════════════════════════════════════════════
all_summary_df = pd.concat(
    [oracle_summary_df, moment_summary_df],
    ignore_index=True
).sort_values(["city", "test_pr_auc"], ascending=[True, False]).reset_index(drop=True)

all_summary_path = f"{MULTICITY_SUMMARY_DIR}/all_models_summary.csv"
all_summary_df.to_csv(all_summary_path, index=False)

print("Gesamtsummary:")
display(all_summary_df[[
    "city", "model", "experiment", "test_pr_auc", "test_f1", "test_prec", "test_recall"
]].sort_values(["city", "test_pr_auc"], ascending=[True, False]))

# Quick Check: Sind die prediction files für spätere Gewichtung vorhanden?
prediction_files = sorted([p for p in os.listdir(MULTICITY_PRED_DIR) if p.endswith(".parquet")])
print(f"\nPrediction-Files: {len(prediction_files)}")
for p in prediction_files[:20]:
    print(" -", p)

print(f"\nAll Summary gespeichert unter: {all_summary_path}")
